# Member 2 — VGG16 Transfer Learning

## Comparative Analysis of Deep Learning Architectures for Multi-Class Brain Tumour MRI Classification

**SE4050 – Deep Learning 2026**

This notebook implements a VGG16 transfer-learning model for classifying brain tumour MRI images into four classes: **glioma**, **meningioma**, **notumor**, and **pituitary**.

The VGG16 backbone is initialised with ImageNet pretrained weights. This is a fundamentally different training approach compared to Member 1's custom CNN (trained from scratch). Therefore, the overall comparison is a comparison of **practical model/training approaches**, not a perfectly isolated architecture-only comparison.

---

### Important Scientific Limitation

Patient-level independence could not be independently verified from the available dataset structure and identifiers.

---
## Phase 1 — Environment Verification and VGG16 Preprocessing Pipeline

**Goal:** Reproduce the environment and verify Member 1's frozen data foundation before any model training.

### Phase 1 Tasks
1. Record environment details
2. Verify split checksums (SHA-256)
3. Download exact Kaggle dataset version 1
4. Load and verify split CSVs
5. Verify class mapping
6. Verify every referenced image exists
7. Build VGG16-specific `tf.data` pipeline with `preprocess_input`
8. Apply common mild augmentation (training only)
9. Verify preprocessed batch shapes and labels
10. Save Phase 1 configuration

### 1.1 — Install Dependencies and Imports

In [ ]:
# Install kagglehub if not present (Colab may not have it by default)
!pip install -q kagglehub

In [ ]:
import os
import sys
import json
import hashlib
import time
import random
import platform
import shutil
import subprocess
import gc

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import sklearn
import kagglehub
import matplotlib
import matplotlib.pyplot as plt

from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess_input

print("Imports completed successfully.")

### 1.2 — Set Random Seeds and Determinism

We set the random seed to 42 for all relevant libraries and enable TensorFlow deterministic operations where possible. Perfect bit-identical GPU reproducibility is not claimed.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Enable deterministic operations where reasonably possible
try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "TensorFlow op determinism enabled"
except Exception as e:
    determinism_status = f"Could not enable op determinism: {e}"

print(f"Seed: {SEED}")
print(f"Determinism: {determinism_status}")

### 1.3 — Record Full Environment

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        gpu_name = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True,
        ).strip().splitlines()[0]
        nvidia_smi = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
            text=True,
        ).strip().splitlines()[0]
    except Exception:
        gpu_name = tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'Unknown GPU')
        nvidia_smi = 'Unavailable'
else:
    gpu_name = 'No GPU detected'
    nvidia_smi = 'N/A'

environment_info = {
    'member': 2,
    'model': 'VGG16',
    'python_version': sys.version,
    'platform': platform.platform(),
    'tensorflow_version': tf.__version__,
    'keras_version': keras.__version__,
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'scikit_learn_version': sklearn.__version__,
    'kagglehub_version': getattr(kagglehub, '__version__', 'unknown'),
    'matplotlib_version': matplotlib.__version__,
    'gpu_name': gpu_name,
    'nvidia_smi': nvidia_smi,
    'seed': SEED,
    'determinism_status': determinism_status,
}

print('Environment Recorded:')
for k, v in environment_info.items():
    print(f'  {k:<22}: {v}')

### 1.4 — Download Exact Kaggle Dataset (Version 1)

We download the **pinned version 1** of the dataset to ensure reproducibility across all group members. This is a critical requirement — newer versions must never be used silently.

In [ ]:
DATASET_HANDLE = "masoudnickparvar/brain-tumor-mri-dataset"
DATASET_VERSION = 1
PINNED_HANDLE = f"{DATASET_HANDLE}/versions/{DATASET_VERSION}"

dataset_path = kagglehub.dataset_download(PINNED_HANDLE)
print(f"Dataset downloaded to: {dataset_path}")
print(f"Contents: {os.listdir(dataset_path)}")

### 1.5 — Define Project Paths

We detect whether the notebook is running on Colab or locally and set paths accordingly.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

# ============================================================
# FRESH-COLAB PROJECT BOOTSTRAP
# ============================================================

REPO_URL = (
    "https://github.com/ramzyhafeel/"
    "brain_tumour_model_comparison.git"
)

PROJECT_ROOT = Path("/content/brain_tumour_model_comparison")


def ensure_project_available():
    """Ensure the complete project repository exists in this Colab runtime."""

    required_items = [
        PROJECT_ROOT / "splits" / "train.csv",
        PROJECT_ROOT / "splits" / "val.csv",
        PROJECT_ROOT / "splits" / "test.csv",
        PROJECT_ROOT / "config" / "class_to_index.json",
        PROJECT_ROOT / "notebooks",
    ]

    if (
        PROJECT_ROOT.exists()
        and all(item.exists() for item in required_items)
    ):
        print("✓ Project repository already available.")
        return PROJECT_ROOT

    print("Project repository not found in this runtime.")
    print("Cloning GitHub repository...")

    # Remove only an incomplete project clone at the expected path.
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)

    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_ROOT)],
        check=True,
    )

    missing = [
        str(item)
        for item in required_items
        if not item.exists()
    ]

    if missing:
        raise FileNotFoundError(
            "Repository cloned, but required project files/folders "
            f"are missing:\n{missing}"
        )

    print("✓ Repository cloned successfully.")
    return PROJECT_ROOT


PROJECT_ROOT = ensure_project_available()

SPLITS_DIR = PROJECT_ROOT / "splits"
CONFIG_DIR = PROJECT_ROOT / "config"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

RESULTS_DIR = PROJECT_ROOT / "results" / "vgg16"
MODELS_DIR = PROJECT_ROOT / "models" / "vgg16"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

print("\n" + "=" * 70)
print("PROJECT SETUP")
print("=" * 70)
print("Project root :", PROJECT_ROOT)
print("Splits       :", SPLITS_DIR)
print("Config       :", CONFIG_DIR)
print("Notebooks    :", NOTEBOOKS_DIR)
print("Results      :", RESULTS_DIR)
print("Models       :", MODELS_DIR)
print("Dataset path :", dataset_path)
print("Working dir  :", Path.cwd())

assert (SPLITS_DIR / "train.csv").is_file()
assert (SPLITS_DIR / "val.csv").is_file()
assert (SPLITS_DIR / "test.csv").is_file()
assert (CONFIG_DIR / "class_to_index.json").is_file()

print("\n✓ PROJECT READY")


### 1.6 — Verify Split Checksums (SHA-256)

Before any model work, we compute the SHA-256 hash of each split CSV and compare against the frozen checksums from Member 1. If any hash differs, we **stop immediately** and do not proceed with training.

In [ ]:
EXPECTED_CHECKSUMS = {
    "train.csv": "7273ef5fc2cb605c5b03b22a23cda0cb383ac0d9ce5949ac3c95649b5a4270cb",
    "val.csv":   "37b24456cfcd1b69df939e36603958eae6a9124b794c32c83a532b9018c6c88f",
    "test.csv":  "9afc40a38949eb4f46f8c9591d3b1f51cdd11aa991bfe2cf1d15c386c5537d39"
}

def compute_sha256(filepath):
    """Compute the SHA-256 hash of a file."""
    sha256 = hashlib.sha256()
    with open(filepath, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            sha256.update(block)
    return sha256.hexdigest()

checksum_ok = True
computed_checksums = {}

for filename, expected_hash in EXPECTED_CHECKSUMS.items():
    filepath = os.path.join(SPLITS_DIR, filename)
    if not os.path.exists(filepath):
        print(f"ERROR: {filepath} does not exist!")
        checksum_ok = False
        continue

    computed = compute_sha256(filepath)
    computed_checksums[filename] = computed
    match = "MATCH" if computed == expected_hash else "MISMATCH"

    if computed != expected_hash:
        checksum_ok = False

    print(f"{filename}: {match}")
    print(f"  Expected:  {expected_hash}")
    print(f"  Computed:  {computed}")
    print()

if not checksum_ok:
    raise RuntimeError(
        "STOP: One or more split checksums do not match! "
        "Do NOT proceed with model training. "
        "Wait for the correct split files."
    )
else:
    print("All split checksums verified successfully.")

### 1.7 — Load Split CSVs and Verify Counts

In [ ]:
train_df = pd.read_csv(os.path.join(SPLITS_DIR, "train.csv"))
val_df   = pd.read_csv(os.path.join(SPLITS_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(SPLITS_DIR, "test.csv"))

EXPECTED_TRAIN = 4353
EXPECTED_VAL   = 1089
EXPECTED_TEST  = 1311

print(f"Train: {len(train_df)} (expected {EXPECTED_TRAIN})")
print(f"Val:   {len(val_df)} (expected {EXPECTED_VAL})")
print(f"Test:  {len(test_df)} (expected {EXPECTED_TEST})")

assert len(train_df) == EXPECTED_TRAIN, f"Train count mismatch: {len(train_df)} vs {EXPECTED_TRAIN}"
assert len(val_df)   == EXPECTED_VAL,   f"Val count mismatch: {len(val_df)} vs {EXPECTED_VAL}"
assert len(test_df)  == EXPECTED_TEST,   f"Test count mismatch: {len(test_df)} vs {EXPECTED_TEST}"

print("\nAll split counts match expected values.")

### 1.8 — Verify Class Mapping

The class-to-index mapping must remain consistent across all four models.

In [ ]:
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
CLASS_TO_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

# Load the frozen class mapping from config
with open(os.path.join(CONFIG_DIR, "class_to_index.json"), "r") as f:
    frozen_mapping = json.load(f)

print("Frozen class mapping:")
for cls, idx in frozen_mapping.items():
    print(f"  {cls} = {idx}")

# Verify they match
assert CLASS_TO_INDEX == frozen_mapping, "Class mapping mismatch!"
print("\nClass mapping verified.")

# Verify class distribution in each split
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{name} class distribution:")
    dist = df["label"].value_counts().sort_index()
    for cls in CLASS_NAMES:
        count = dist.get(cls, 0)
        print(f"  {cls}: {count}")

# Verify all labels are valid
all_labels = set(train_df["label"]) | set(val_df["label"]) | set(test_df["label"])
assert all_labels == set(CLASS_NAMES), f"Unexpected labels found: {all_labels - set(CLASS_NAMES)}"
print("\nAll labels are valid class names.")

### 1.9 — Verify Every Referenced Image Exists

We check that every filepath referenced in the split CSVs corresponds to an actual image file in the downloaded dataset.

In [ ]:
missing_files = []

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for _, row in df.iterrows():
        full_path = os.path.join(dataset_path, row["filepath"])
        if not os.path.isfile(full_path):
            missing_files.append((name, row["filepath"]))

if missing_files:
    print(f"ERROR: {len(missing_files)} referenced files are missing!")
    for split, path in missing_files[:10]:
        print(f"  [{split}] {path}")
    raise RuntimeError("Missing image files detected. Cannot proceed.")
else:
    total = len(train_df) + len(val_df) + len(test_df)
    print(f"All {total} referenced images verified to exist.")

---
## VGG16-Specific Preprocessing Pipeline

### Why `preprocess_input` instead of `Rescaling(1./255)`?

The VGG16 model was originally trained on ImageNet with a specific preprocessing scheme:
- Converts images from RGB to BGR
- Subtracts the ImageNet channel means (approximately [103.939, 116.779, 123.68])

This is fundamentally different from simple 0-1 rescaling. Using `Rescaling(1./255)` followed by `preprocess_input` would result in **inappropriate double normalisation** and degraded performance.

Therefore, the pipeline decodes images to float values in approximately [0, 255], and then applies `keras.applications.vgg16.preprocess_input` to transform them.

The Custom CNN (Member 1) uses `Rescaling(1./255)` because it was trained from scratch without pretrained weights.

### 1.10 — Define Pipeline Constants

In [ ]:
IMAGE_HEIGHT = 224
IMAGE_WIDTH  = 224
CHANNELS     = 3
IMAGE_SIZE   = (IMAGE_HEIGHT, IMAGE_WIDTH)
INPUT_SHAPE  = (IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS)
BATCH_SIZE   = 16
NUM_CLASSES  = 4
MAX_EPOCHS   = 20

print(f"Image size:  {IMAGE_HEIGHT}x{IMAGE_WIDTH}x{CHANNELS}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Num classes: {NUM_CLASSES}")
print(f"Max epochs:  {MAX_EPOCHS}")

### 1.11 — Define Data Augmentation Layer

Augmentation is applied **only during training** using a `tf.keras.Sequential` model. The augmentation is intentionally conservative because these are medical MRI images:

- **RandomRotation** (factor = 0.03): approximately +/-10.8 degrees
- **RandomZoom** (height/width = +/-0.08)
- **RandomTranslation** (height/width = 0.05)

In [ ]:
def make_augmentation(seed=SEED):
    return tf.keras.Sequential([
        tf.keras.layers.RandomRotation(factor=0.03, seed=seed),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.08, 0.08),
            width_factor=(-0.08, 0.08),
            seed=seed,
        ),
        tf.keras.layers.RandomTranslation(
            height_factor=0.05,
            width_factor=0.05,
            seed=seed,
        ),
    ], name='augmentation')

# Preview/smoke instance. Each controlled experiment gets a fresh instance.
augmentation_layer = make_augmentation(SEED)
print('Training-only augmentation factory defined.')

### 1.12 — Build `tf.data` Pipeline

The pipeline:
1. Reads image file paths and labels from the split CSVs
2. Decodes JPEG images to float32 tensors in [0, 255]
3. Resizes to 224x224x3
4. For training: applies augmentation, then `vgg16_preprocess_input`
5. For validation/test: applies `vgg16_preprocess_input` only (no augmentation)
6. Uses `tf.data.AUTOTUNE` for prefetching

In [ ]:
def decode_and_resize(filepath, label):
    """Decode JPEG, resize to 224x224 and keep float32 values in [0,255]."""
    raw = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(raw, channels=CHANNELS)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label

def build_dataset(df, is_training=False):
    """Build raw-image tf.data pipeline. Model owns augmentation + preprocessing."""
    filepaths = [os.path.join(str(dataset_path), fp) for fp in df['filepath'].values]
    labels = [CLASS_TO_INDEX[lbl] for lbl in df['label'].values]
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if is_training:
        ds = ds.shuffle(
            buffer_size=len(filepaths),
            seed=SEED,
            reshuffle_each_iteration=True,
        )
    ds = ds.map(decode_and_resize, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print('VGG16 dataset builder defined.')
print('tf.data keeps pixels in raw [0,255] range; augmentation and VGG16 preprocess_input run exactly once inside the model.')

### 1.13 — Create Training and Validation Datasets

In [ ]:
train_ds = build_dataset(train_df, is_training=True)
val_ds   = build_dataset(val_df,   is_training=False)

print(f"Training dataset:   {tf.data.experimental.cardinality(train_ds).numpy()} batches")
print(f"Validation dataset: {tf.data.experimental.cardinality(val_ds).numpy()} batches")

### 1.14 — Verify Pipeline Output

We verify:
- Batch image shape is `(16, 224, 224, 3)`
- Labels are integer indices in `{0, 1, 2, 3}`
- Four distinct classes are present in the dataset
- Preprocessed pixel values are NOT in [0, 1] (VGG16 preprocess_input subtracts means, producing negative values)
- Validation is unshuffled (first labels match the expected order from the CSV)

In [ ]:
# Verify that tf.data returns RAW float32 images in [0,255].
# VGG16 preprocessing intentionally happens exactly once INSIDE the model.

for images, labels in train_ds.take(1):
    raw_min = float(tf.reduce_min(images).numpy())
    raw_max = float(tf.reduce_max(images).numpy())
    raw_mean = float(tf.reduce_mean(images).numpy())

    print(f"Training batch image shape: {images.shape}")
    print(f"Training batch label shape: {labels.shape}")
    print(f"Training batch image dtype: {images.dtype}")
    print(f"Training batch label dtype: {labels.dtype}")
    print(f"Raw image min:  {raw_min:.2f}")
    print(f"Raw image max:  {raw_max:.2f}")
    print(f"Raw image mean: {raw_mean:.2f}")
    print(f"Labels in batch: {labels.numpy()}")

assert images.shape[1:] == (224, 224, 3), "Unexpected image shape."
assert raw_min >= 0.0, "Raw tf.data image values should not be negative."
assert raw_max <= 255.0, "Raw tf.data image values should remain <= 255."

# Probe the architecture-specific transform separately.
processed_probe = vgg16_preprocess_input(tf.identity(images))
processed_min = float(tf.reduce_min(processed_probe).numpy())
processed_max = float(tf.reduce_max(processed_probe).numpy())

print(f"\nVGG16 processed probe min: {processed_min:.2f}")
print(f"VGG16 processed probe max: {processed_max:.2f}")

assert processed_min < 0.0, (
    "VGG16 preprocess_input should create negative values after ImageNet "
    "mean subtraction."
)

print("\n✓ tf.data returns raw [0,255] images.")
print("✓ vgg16_preprocess_input transforms them correctly.")
print("✓ The model will apply augmentation and preprocessing exactly once.")


In [ ]:
# Verify all four classes appear across the full training set
all_train_labels = []
for _, labels_batch in train_ds:
    all_train_labels.extend(labels_batch.numpy().tolist())
unique_train = sorted(set(all_train_labels))
print(f"Unique training labels: {unique_train}")
assert unique_train == [0, 1, 2, 3], "Not all four classes found in training set!"
print("All four classes confirmed in training data.")

In [ ]:
# Verify validation is unshuffled: first labels should match CSV order
expected_first_val_labels = [CLASS_TO_INDEX[lbl] for lbl in val_df["label"].values[:BATCH_SIZE]]
for _, labels_batch in val_ds.take(1):
    actual_first_val_labels = labels_batch.numpy().tolist()

print(f"Expected first {BATCH_SIZE} val labels: {expected_first_val_labels}")
print(f"Actual first {BATCH_SIZE} val labels:   {actual_first_val_labels}")
assert expected_first_val_labels == actual_first_val_labels, "Validation dataset is shuffled!"
print("\nValidation dataset confirmed unshuffled.")

### 1.15 — Verify Test Manifest (No Model Evaluation)

We verify that the test CSV references valid images but do **NOT** create a test `tf.data.Dataset` for model evaluation at this phase.

In [ ]:
print("Test manifest verification:")
print(f"  Total test samples: {len(test_df)}")
print(f"  Test class distribution:")
for cls in CLASS_NAMES:
    count = len(test_df[test_df['label'] == cls])
    print(f"    {cls}: {count}")

# Verify all test images exist (already checked above, but confirm explicitly)
test_missing = 0
for _, row in test_df.iterrows():
    if not os.path.isfile(os.path.join(dataset_path, row["filepath"])):
        test_missing += 1

print(f"  Missing test images: {test_missing}")
assert test_missing == 0, "Some test images are missing!"
print("\nTest manifest verified. No test dataset created for model evaluation.")

### 1.16 — Pipeline Demonstration: Sample Visualisation

In [ ]:
raw_images, demo_labels = next(iter(train_ds))
demo_aug = make_augmentation(SEED)
augmented_images = demo_aug(raw_images, training=True)
processed_images = vgg16_preprocess_input(augmented_images)

# Display augmented raw RGB images (not the normalized tensor) for interpretability.
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('VGG16 Pipeline Demo — Training-only augmentation preview', fontsize=14)
for i, ax in enumerate(axes.flat):
    if i >= raw_images.shape[0]:
        break
    vis = tf.clip_by_value(augmented_images[i], 0, 255).numpy().astype(np.uint8)
    ax.imshow(vis)
    ax.set_title(f'{CLASS_NAMES[int(demo_labels[i])]} ({int(demo_labels[i])})')
    ax.axis('off')
plt.tight_layout()
demo_plot_path = os.path.join(RESULTS_DIR, 'phase1_pipeline_demo.png')
plt.savefig(demo_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print('Pipeline demo saved to:', demo_plot_path)
print('Processed probe range:', float(tf.reduce_min(processed_images)), 'to', float(tf.reduce_max(processed_images)))

### 1.17 — Save Phase 1 Configuration and Environment

In [ ]:
phase1_config = {
    "member": 2,
    "model": "VGG16",
    "phase": 1,
    "dataset_handle": DATASET_HANDLE,
    "dataset_version": DATASET_VERSION,
    "pinned_dataset_handle": PINNED_HANDLE,
    "dataset_path": dataset_path,
    "seed": SEED,
    "image_size": f"{IMAGE_HEIGHT}x{IMAGE_WIDTH}x{CHANNELS}",
    "batch_size": BATCH_SIZE,
    "num_classes": NUM_CLASSES,
    "max_epochs": MAX_EPOCHS,
    "class_names": CLASS_NAMES,
    "class_to_index": CLASS_TO_INDEX,
    "preprocessing": "keras.applications.vgg16.preprocess_input (NO Rescaling 1/255)",
    "augmentation": {
        "rotation_factor": 0.03,
        "zoom_range": [-0.08, 0.08],
        "translation_factor": 0.05,
        "applied_to": "training_only"
    },
    "split_counts": {
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df)
    },
    "split_checksums": computed_checksums,
    "split_checksums_verified": True,
    "all_images_verified": True,
    "test_dataset_created_for_evaluation": False,
    "patient_level_split_verified": False,
    "transfer_learning_limitation": (
        "VGG16 uses ImageNet pretrained weights. The comparison with the Custom CNN "
        "(trained from scratch) is therefore a comparison of practical model/training "
        "approaches, not a perfectly isolated architecture-only comparison."
    ),
    "patient_level_limitation": (
        "Patient-level independence could not be independently verified from "
        "the available dataset structure and identifiers."
    )
}

config_path = os.path.join(RESULTS_DIR, "phase1_config.json")
with open(config_path, "w") as f:
    json.dump(phase1_config, f, indent=4)
print(f"Phase 1 config saved to: {config_path}")

env_path = os.path.join(RESULTS_DIR, "vgg16_environment.json")
with open(env_path, "w") as f:
    json.dump(environment_info, f, indent=4)
print(f"Environment info saved to: {env_path}")

---
## Phase 1 — Complete

### Summary

Phase 1 of Member 2 (VGG16) has established the verified data foundation:

1. **Environment recorded** — Python, TensorFlow, Keras, NumPy, Pandas, scikit-learn, KaggleHub, GPU, seed
2. **Split checksums verified** — All three SHA-256 hashes match Member 1's frozen manifests
3. **Dataset downloaded** — Exact Kaggle version 1
4. **Split counts verified** — Train=4353, Val=1089, Test=1311
5. **Class mapping verified** — glioma=0, meningioma=1, notumor=2, pituitary=3
6. **All images verified** — Every referenced file exists
7. **VGG16 tf.data pipeline built** — Uses `vgg16.preprocess_input`, NOT `Rescaling(1./255)`
8. **Augmentation applied** — Training-only, conservative (rotation/zoom/translation)
9. **Pipeline verified** — Correct shapes, correct label types, four classes, val unshuffled
10. **No test evaluation performed** — Test manifest verified only
11. **Phase 1 config and environment saved**

---
## Phase 2 — Frozen VGG16 Baseline and Smoke Test

**Goal:** Build a VGG16 transfer-learning model with a frozen ImageNet backbone and a modest classifier head. Run a 1-epoch smoke test to verify the pipeline is fully functional before controlled experiments.

### Architecture Design

```
Input (224x224x3)
  -> Augmentation (training only)
  -> vgg16.preprocess_input
  -> VGG16 backbone (frozen, include_top=False, ImageNet weights)
  -> GlobalAveragePooling2D
  -> Dense(256, ReLU)
  -> Dropout(0.3)
  -> Dense(4, softmax)
```

### Transfer Learning Approach

**Why ImageNet transfer learning?**
VGG16 was trained on ImageNet (1.2M images, 1000 classes). The lower convolutional layers learn general visual features (edges, textures, shapes) that transfer well to medical imaging tasks. Starting from these pretrained weights allows the model to converge faster and potentially achieve higher accuracy with our relatively small brain tumour dataset (4353 training images) compared to training from scratch.

**Why freeze the backbone initially?**
Freezing the pretrained convolutional layers prevents catastrophic forgetting — if we train the entire network with a high learning rate, the carefully learned ImageNet features would be destroyed. We first train only the classifier head to adapt to our 4-class task.

### Layer Justifications

| Layer | Justification |
|-------|---------------|
| **GlobalAveragePooling2D** | Reduces VGG16's (7x7x512) output to a (512,) vector by averaging each feature map. Preferred over `Flatten` because Flatten would produce a 25088-dimensional vector, leading to an excessively large Dense layer and increased overfitting risk. GAP also provides spatial invariance. |
| **Dense(256, ReLU)** | Provides a modest non-linear transformation to learn task-specific feature combinations. 256 units balance capacity with overfitting risk. ReLU is the standard hidden-layer activation — computationally efficient and avoids vanishing gradient. |
| **Dropout(0.3)** | Regularisation to reduce overfitting. During training, 30% of connections are randomly dropped, encouraging the network to learn redundant representations. 0.3 is a conservative starting point. |
| **Dense(4, softmax)** | Output layer with one unit per class. Softmax ensures outputs sum to 1, producing a valid probability distribution over the four tumour classes. |

### Training Configuration Justifications

| Setting | Choice | Justification |
|---------|--------|---------------|
| **Loss** | SparseCategoricalCrossentropy | Standard loss for multi-class classification with integer labels. 'Sparse' variant avoids one-hot encoding overhead. |
| **Optimizer** | Adam (lr=0.001) | Adam combines momentum and adaptive learning rates, providing robust convergence. 0.001 is the default and works well for training a randomly initialised head on top of frozen features. |
| **Softmax** | Output activation | Required for multi-class classification to produce normalised class probabilities. |

### 2.1 — Build VGG16 Base Model

In [ ]:
# Load VGG16 with ImageNet weights, without the top classification layers
base_model = tf.keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE
)

# Freeze all backbone layers
base_model.trainable = False

print(f"VGG16 backbone loaded with ImageNet weights.")
print(f"Backbone trainable: {base_model.trainable}")
print(f"Number of backbone layers: {len(base_model.layers)}")
print(f"Backbone output shape: {base_model.output_shape}")
print()
base_model.summary()

### 2.2 — Build Complete Model with Classifier Head

In [ ]:
def build_vgg16_model(base_model, augmentation_layer, dropout_rate=0.3,
                      dense_units=256, num_classes=NUM_CLASSES,
                      model_name="VGG16"):
    """Build a VGG16 transfer-learning model with a classifier head.

    Architecture:
        Input -> Augmentation -> preprocess_input -> VGG16 -> GAP ->
        Dense(dense_units, ReLU) -> Dropout -> Dense(num_classes, softmax)

    Args:
        base_model: Frozen VGG16 backbone.
        augmentation_layer: Keras Sequential augmentation layer.
        dropout_rate: Dropout probability for regularisation.
        dense_units: Number of units in the hidden Dense layer.
        num_classes: Number of output classes.
        model_name: Name for the model.

    Returns:
        A compiled-ready tf.keras.Model.
    """
    inputs = tf.keras.Input(shape=INPUT_SHAPE, name="input_image")

    # Augmentation (only active during training)
    x = augmentation_layer(inputs)

    # VGG16-specific preprocessing: RGB to BGR, subtract ImageNet means
    # Input values must be in [0, 255] — no prior Rescaling(1./255)
    x = vgg16_preprocess_input(x)

    # Frozen VGG16 feature extractor
    x = base_model(x, training=False)

    # Classification head
    x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = tf.keras.layers.Dense(dense_units, activation="relu", name="classifier_dense")(x)
    x = tf.keras.layers.Dropout(dropout_rate, name="classifier_dropout")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="classifier_output")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name=model_name)
    return model


model = build_vgg16_model(base_model, augmentation_layer, dropout_rate=0.3)
model.summary()

### 2.3 — Verify Model Properties

In [ ]:
# Count trainable and non-trainable parameters
total_params = model.count_params()
trainable_params = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
non_trainable_params = sum(tf.keras.backend.count_params(w) for w in model.non_trainable_weights)

print(f"Total parameters:         {total_params:,}")
print(f"Trainable parameters:     {trainable_params:,}")
print(f"Non-trainable parameters: {non_trainable_params:,}")

# Verify backbone is frozen
print(f"\nBackbone trainable: {base_model.trainable}")
backbone_trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Backbone layers with trainable=True: {backbone_trainable_count}")

# Verify input shape
print(f"\nModel input shape:  {model.input_shape}")
print(f"Model output shape: {model.output_shape}")
assert model.input_shape == (None, 224, 224, 3), "Input shape mismatch!"
assert model.output_shape == (None, 4), "Output shape mismatch!"
print("\nInput (224x224x3) and output (4 classes) shapes verified.")

### 2.4 — Verify Softmax Output Sums to 1

In [ ]:
# Run a single batch through the model to verify softmax outputs
for images, labels in val_ds.take(1):
    preds = model(images, training=False)
    print(f"Predictions shape: {preds.shape}")
    print(f"First 5 prediction rows:")
    for i in range(min(5, preds.shape[0])):
        row = preds[i].numpy()
        print(f"  [{', '.join(f'{v:.4f}' for v in row)}]  sum={sum(row):.6f}")

    # Verify all softmax outputs sum to approximately 1.0
    sums = tf.reduce_sum(preds, axis=1).numpy()
    assert np.allclose(sums, 1.0, atol=1e-5), "Softmax outputs do not sum to 1!"
    print(f"\nAll {preds.shape[0]} softmax outputs sum to ~1.0. Verified.")

### 2.5 — Compile the Model

In [ ]:
LEARNING_RATE = 0.001

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

print(f"Model compiled.")
print(f"  Optimizer:     Adam")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Loss:          SparseCategoricalCrossentropy")
print(f"  Metrics:       accuracy")

### 2.6 — Set Up Callbacks for Smoke Test

In [ ]:
# Create checkpoint directory for smoke test
SMOKE_DIR = os.path.join(RESULTS_DIR, "phase2_smoke")
os.makedirs(SMOKE_DIR, exist_ok=True)

smoke_checkpoint_path = os.path.join(SMOKE_DIR, "vgg16_smoke.keras")

smoke_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=smoke_checkpoint_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

print(f"Callbacks configured.")
print(f"  Checkpoint path: {smoke_checkpoint_path}")
print(f"  EarlyStopping: monitor=val_loss, patience=4, restore_best_weights=True")

### 2.7 — Run 1-Epoch Smoke Test

This is a **smoke test only** — we run exactly 1 epoch to verify:
- Training forward/backward pass works without errors
- Validation evaluation works
- Checkpoint saving works
- No NaN values appear in loss or accuracy

**This result is NOT reported as final performance.**

In [ ]:
SMOKE_EPOCHS = 1

print(f"Running {SMOKE_EPOCHS}-epoch smoke test...")
print("=" * 60)

smoke_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=SMOKE_EPOCHS,
    callbacks=smoke_callbacks,
    verbose=1
)

print("=" * 60)
print("Smoke test complete.")

### 2.8 — Verify Smoke Test Results

In [ ]:
# Extract smoke test metrics
smoke_train_loss = smoke_history.history["loss"][0]
smoke_train_acc  = smoke_history.history["accuracy"][0]
smoke_val_loss   = smoke_history.history["val_loss"][0]
smoke_val_acc    = smoke_history.history["val_accuracy"][0]

print("Smoke Test Results (1 epoch — NOT final performance):")
print(f"  Training Loss:     {smoke_train_loss:.4f}")
print(f"  Training Accuracy: {smoke_train_acc:.4f}")
print(f"  Val Loss:          {smoke_val_loss:.4f}")
print(f"  Val Accuracy:      {smoke_val_acc:.4f}")

# Verify no NaN values
assert not np.isnan(smoke_train_loss), "Training loss is NaN!"
assert not np.isnan(smoke_train_acc),  "Training accuracy is NaN!"
assert not np.isnan(smoke_val_loss),   "Validation loss is NaN!"
assert not np.isnan(smoke_val_acc),    "Validation accuracy is NaN!"
print("\nNo NaN values detected. Pipeline is functional.")

# Verify checkpoint was saved
assert os.path.exists(smoke_checkpoint_path), "Checkpoint was not saved!"
checkpoint_size_mb = os.path.getsize(smoke_checkpoint_path) / (1024 * 1024)
print(f"Checkpoint saved: {smoke_checkpoint_path} ({checkpoint_size_mb:.2f} MB)")

# Verify test set is untouched
print("\nTest set status: UNTOUCHED (no test evaluation performed).")

### 2.9 — Smoke Test Visualisation

In [ ]:
# Simple bar chart showing the single-epoch smoke results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].bar(["Train", "Val"], [smoke_train_loss, smoke_val_loss],
            color=["steelblue", "darkorange"])
axes[0].set_title("Smoke Test — Loss (1 Epoch)")
axes[0].set_ylabel("Loss")
for i, v in enumerate([smoke_train_loss, smoke_val_loss]):
    axes[0].text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=10)

# Accuracy
axes[1].bar(["Train", "Val"], [smoke_train_acc, smoke_val_acc],
            color=["steelblue", "darkorange"])
axes[1].set_title("Smoke Test — Accuracy (1 Epoch)")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim([0, 1])
for i, v in enumerate([smoke_train_acc, smoke_val_acc]):
    axes[1].text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=10)

plt.suptitle("VGG16 Smoke Test Results (NOT final performance)", fontsize=13, y=1.02)
plt.tight_layout()

smoke_plot_path = os.path.join(SMOKE_DIR, "smoke_test_results.png")
plt.savefig(smoke_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Smoke test plot saved to: {smoke_plot_path}")

### 2.10 — Save Phase 2 Configuration and Smoke History

In [ ]:
# Save smoke test history as CSV
smoke_history_df = pd.DataFrame(smoke_history.history)
smoke_history_path = os.path.join(SMOKE_DIR, "smoke_history.csv")
smoke_history_df.to_csv(smoke_history_path, index=False)
print(f"Smoke history saved to: {smoke_history_path}")

# Save Phase 2 configuration
phase2_config = {
    "member": 2,
    "model": "VGG16",
    "phase": 2,
    "description": "Frozen VGG16 baseline with 1-epoch smoke test",
    "backbone": {
        "architecture": "VGG16",
        "weights": "imagenet",
        "include_top": False,
        "trainable": False,
        "total_layers": len(base_model.layers),
        "backbone_params": sum(tf.keras.backend.count_params(w) for w in base_model.weights)
    },
    "classifier_head": {
        "pooling": "GlobalAveragePooling2D",
        "dense_units": 256,
        "dense_activation": "relu",
        "dropout_rate": 0.3,
        "output_units": NUM_CLASSES,
        "output_activation": "softmax"
    },
    "training": {
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "loss": "SparseCategoricalCrossentropy",
        "batch_size": BATCH_SIZE,
        "smoke_epochs": SMOKE_EPOCHS
    },
    "parameters": {
        "total": int(total_params),
        "trainable": int(trainable_params),
        "non_trainable": int(non_trainable_params)
    },
    "smoke_results": {
        "train_loss": float(smoke_train_loss),
        "train_accuracy": float(smoke_train_acc),
        "val_loss": float(smoke_val_loss),
        "val_accuracy": float(smoke_val_acc),
        "nan_detected": False,
        "checkpoint_saved": True,
        "checkpoint_size_mb": float(checkpoint_size_mb)
    },
    "test_evaluation_performed": False,
    "seed": SEED
}

config_path = os.path.join(SMOKE_DIR, "phase2_config.json")
with open(config_path, "w") as f:
    json.dump(phase2_config, f, indent=4)
print(f"Phase 2 config saved to: {config_path}")

### 2.11 — Save Model Summary to Text

In [ ]:
# Save the model summary to a text file for reference
summary_lines = []
model.summary(print_fn=lambda x: summary_lines.append(x))
summary_text = "\n".join(summary_lines)

summary_path = os.path.join(SMOKE_DIR, "model_summary.txt")
with open(summary_path, "w") as f:
    f.write(summary_text)
print(f"Model summary saved to: {summary_path}")
print()
print(summary_text)

---
## Phase 2 — Complete

### Summary

Phase 2 of Member 2 (VGG16) built and smoke-tested the frozen transfer-learning baseline:

1. **VGG16 backbone loaded** — ImageNet weights, `include_top=False`, `input_shape=(224,224,3)`
2. **Backbone frozen** — `base_model.trainable = False`, all 14,714,688 parameters non-trainable
3. **Classifier head built** — GAP -> Dense(256, ReLU) -> Dropout(0.3) -> Dense(4, softmax)
4. **Model compiled** — Adam (lr=0.001), SparseCategoricalCrossentropy
5. **Softmax verified** — All outputs sum to ~1.0
6. **1-epoch smoke test passed** — No NaN, training and validation work correctly
7. **Checkpoint saved** — `results/vgg16/phase2_smoke/vgg16_smoke.keras`
8. **Test set untouched** — No test evaluation performed
9. **Phase 2 config and history saved**

**Note:** The 1-epoch smoke results are NOT reported as final model performance.

---
## Phase 3 — Controlled Validation-Only VGG16 Experiments

**Goal:** Conduct scientifically controlled transfer-learning experiments on VGG16 using **validation data only** to select the final model configuration.

### Strict Scientific Rules for Phase 3
1. **Held-Out Test Set Rule:** The test set (`test.csv`) is **strictly held out**. It must NEVER influence model selection, architecture design, dropout rate, learning rate, fine-tuning depth, or epoch selection. Not a single test image is loaded or evaluated in this phase.
2. **Predefined Selection Rule:**
   - **Primary criterion:** Minimum validation loss (`best_val_loss`).
   - **Secondary criterion:** Maximum validation Macro F1 (`validation_macro_f1`).
3. **Fair Comparison Protocol:** All experiments use the exact same image size (`224x224x3`), batch size (`16`), random seed (`42`), loss function (`SparseCategoricalCrossentropy`), and maximum epoch budget (`20 epochs` with EarlyStopping patience `4` monitoring `val_loss`).

### Experiment Matrix

| Experiment ID | Architecture & Backbone | Head Configuration | Optimizer & LR | Purpose |
|---|---|---|---|---|
| **VGG16-A** | VGG16 (ImageNet, frozen) | GAP + Dense(256, ReLU) + **Dropout(0.30)** + Dense(4, softmax) | Adam, $\eta=10^{-3}$ | Baseline transfer learning with standard regularization |
| **VGG16-B** | VGG16 (ImageNet, frozen) | GAP + Dense(256, ReLU) + **Dropout(0.50)** + Dense(4, softmax) | Adam, $\eta=10^{-3}$ | Controlled regularization study: evaluate impact of higher dropout on overfitting |
| **VGG16-C** | VGG16 (ImageNet, **unfrozen Block 5**) | GAP + Dense(256, ReLU) + Best Dropout + Dense(4, softmax) | Adam, $\eta=10^{-5}$ | Controlled fine-tuning study: adapt top conv block with small learning rate |

### 3.1 — Setup Phase 3 Output Directories and Utilities

In [ ]:
import shutil
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
import seaborn as sns

# Define Phase 3 subdirectories
PHASE3_DIR = os.path.join(RESULTS_DIR, "phase3_experiments")
CHECKPOINTS_DIR = os.path.join(PHASE3_DIR, "checkpoints")
CONFIGS_DIR = os.path.join(PHASE3_DIR, "configs")
HISTORIES_DIR = os.path.join(PHASE3_DIR, "histories")
PLOTS_DIR = os.path.join(PHASE3_DIR, "plots")
PREDICTIONS_DIR = os.path.join(PHASE3_DIR, "validation_predictions")

for directory in [PHASE3_DIR, CHECKPOINTS_DIR, CONFIGS_DIR, HISTORIES_DIR, PLOTS_DIR, PREDICTIONS_DIR]:
    os.makedirs(directory, exist_ok=True)

print("Phase 3 output directories verified:")
print(f"  Phase 3 root: {PHASE3_DIR}")
print(f"  Checkpoints:  {CHECKPOINTS_DIR}")
print(f"  Histories:    {HISTORIES_DIR}")
print(f"  Plots:        {PLOTS_DIR}")
print(f"  Predictions:  {PREDICTIONS_DIR}")

### 3.2 — Define Evaluation and Plotting Helper Functions

To guarantee consistency across all experiments, we implement standard helper functions:
1. `plot_learning_curves`: Plots training and validation loss and accuracy, highlighting the best epoch.
2. `evaluate_validation_performance`: Generates unshuffled validation predictions, calculates macro and weighted metrics, and exports a detailed prediction manifest.

In [ ]:
def plot_learning_curves(history, exp_id, save_path=None):
    """Plot training and validation loss and accuracy curves with best epoch marker."""
    train_loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    train_acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    epochs_range = range(1, len(train_loss) + 1)
    best_epoch = int(np.argmin(val_loss)) + 1
    best_val_loss_val = float(np.min(val_loss))
    best_val_acc_val = float(val_acc[best_epoch - 1])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curve
    ax1.plot(epochs_range, train_loss, 'o-', color='#1f77b4', label='Training Loss', linewidth=2)
    ax1.plot(epochs_range, val_loss, 's--', color='#ff7f0e', label='Validation Loss', linewidth=2)
    ax1.axvline(best_epoch, color='crimson', linestyle=':', label=f'Best Epoch ({best_epoch})')
    ax1.scatter([best_epoch], [best_val_loss_val], color='crimson', s=100, zorder=5)
    ax1.set_title(f'{exp_id} — Loss Curve', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch', fontsize=11)
    ax1.set_ylabel('Loss (Sparse Categorical Crossentropy)', fontsize=11)
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend(loc='upper right', frameon=True)

    # Accuracy curve
    ax2.plot(epochs_range, train_acc, 'o-', color='#1f77b4', label='Training Accuracy', linewidth=2)
    ax2.plot(epochs_range, val_acc, 's--', color='#2ca02c', label='Validation Accuracy', linewidth=2)
    ax2.axvline(best_epoch, color='crimson', linestyle=':', label=f'Best Epoch ({best_epoch})')
    ax2.scatter([best_epoch], [best_val_acc_val], color='crimson', s=100, zorder=5)
    ax2.set_title(f'{exp_id} — Accuracy Curve', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch', fontsize=11)
    ax2.set_ylabel('Accuracy', fontsize=11)
    ax2.set_ylim([0.0, 1.02])
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend(loc='lower right', frameon=True)

    plt.suptitle(f'Learning Curves for Experiment {exp_id}', fontsize=14, y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return best_epoch, best_val_loss_val, best_val_acc_val


def evaluate_validation_performance(model, val_dataset, val_dataframe, exp_id):
    """Evaluate model on unshuffled validation set and compute scikit-learn metrics."""
    # Extract predictions (val_dataset is strictly unshuffled)
    val_probabilities = model.predict(val_dataset, verbose=0)
    val_predictions = np.argmax(val_probabilities, axis=1)
    val_confidences = np.max(val_probabilities, axis=1)

    # True labels from frozen manifest
    val_true_labels = np.array([CLASS_TO_INDEX[label] for label in val_dataframe['label'].values])

    # Accuracy & Scikit-learn macro metrics
    val_accuracy = float(accuracy_score(val_true_labels, val_predictions))
    macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(
        val_true_labels, val_predictions, average='macro', zero_division=0
    )
    weighted_prec, weighted_rec, weighted_f1, _ = precision_recall_fscore_support(
        val_true_labels, val_predictions, average='weighted', zero_division=0
    )

    # Save validation predictions manifest
    pred_df = pd.DataFrame({
        'filepath': val_dataframe['filepath'].values,
        'true_label': val_dataframe['label'].values,
        'true_index': val_true_labels,
        'predicted_index': val_predictions,
        'predicted_label': [CLASS_NAMES[idx] for idx in val_predictions],
        'confidence': val_confidences,
        'correct': (val_true_labels == val_predictions),
        'prob_glioma': val_probabilities[:, 0],
        'prob_meningioma': val_probabilities[:, 1],
        'prob_notumor': val_probabilities[:, 2],
        'prob_pituitary': val_probabilities[:, 3]
    })
    pred_save_path = os.path.join(PREDICTIONS_DIR, f"{exp_id.lower()}_val_predictions.csv")
    pred_df.to_csv(pred_save_path, index=False)
    print(f"  Validation predictions saved to: {pred_save_path}")

    results = {
        'validation_accuracy': val_accuracy,
        'validation_macro_precision': float(macro_prec),
        'validation_macro_recall': float(macro_rec),
        'validation_macro_f1': float(macro_f1),
        'validation_weighted_precision': float(weighted_prec),
        'validation_weighted_recall': float(weighted_rec),
        'validation_weighted_f1': float(weighted_f1),
        'y_true': val_true_labels,
        'y_pred': val_predictions,
        'y_prob': val_probabilities
    }
    return results

print("Evaluation helper functions defined.")

### 3.3 — Experiment VGG16-A: Baseline Frozen Backbone with Dropout 0.30

**Hypothesis:** Freezing the VGG16 convolutional backbone and training a modest classification head (GAP + Dense 256 + Dropout 0.30) provides stable transfer learning while mitigating overfitting on a 4,353-sample training set.

**Configuration:**
- Backbone: VGG16 (`include_top=False`, `weights='imagenet'`, frozen)
- Head: GAP -> Dense(256, ReLU) -> Dropout(0.30) -> Dense(4, softmax)
- Optimizer: Adam, Initial Learning Rate = 0.001
- Budget: 20 epochs, EarlyStopping patience = 4 on `val_loss`

In [ ]:
globals().pop('model', None)
globals().pop('base_model', None)
tf.keras.backend.clear_session()
gc.collect()
EXP_A_ID = "VGG16_A"
print(f"\n{'='*70}\nStarting Experiment {EXP_A_ID}\n{'='*70}")

# Reset seeds for reproducibility
tf.keras.utils.set_random_seed(SEED)

# Instantiate fresh base model
base_vgg16_a = tf.keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE
)
base_vgg16_a.trainable = False

# Build model with Dropout 0.30
model_vgg16_a = build_vgg16_model(
    base_model=base_vgg16_a,
    augmentation_layer=make_augmentation(SEED),
    dropout_rate=0.30,
    dense_units=256,
    num_classes=NUM_CLASSES,
    model_name=EXP_A_ID
)

# Compile model
LR_A = 0.001
model_vgg16_a.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_A),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

# Count parameters
params_a_total = model_vgg16_a.count_params()
params_a_trainable = sum(tf.keras.backend.count_params(w) for w in model_vgg16_a.trainable_weights)
params_a_nontrainable = sum(tf.keras.backend.count_params(w) for w in model_vgg16_a.non_trainable_weights)

print(f"Model: {EXP_A_ID}")
print(f"  Total parameters:         {params_a_total:,}")
print(f"  Trainable parameters:     {params_a_trainable:,}")
print(f"  Non-trainable parameters: {params_a_nontrainable:,}")

# Setup checkpoint and early stopping
checkpoint_a_path = os.path.join(CHECKPOINTS_DIR, "vgg16_a_checkpoint.keras")
callbacks_a = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_a_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

# Train model
start_time_a = time.time()
history_vgg16_a = model_vgg16_a.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks_a,
    verbose=1
)
duration_a = time.time() - start_time_a

# Save history
hist_a_df = pd.DataFrame(history_vgg16_a.history)
hist_a_path = os.path.join(HISTORIES_DIR, "vgg16_a_history.csv")
hist_a_df.to_csv(hist_a_path, index=False)
print(f"History saved to: {hist_a_path}")

# Plot learning curves
plot_a_path = os.path.join(PLOTS_DIR, "vgg16_a_learning_curves.png")
best_epoch_a, best_val_loss_a, best_val_acc_a = plot_learning_curves(history_vgg16_a, EXP_A_ID, plot_a_path)

# Evaluate on validation set
eval_a = evaluate_validation_performance(model_vgg16_a, val_ds, val_df, EXP_A_ID)
train_loss_best_a = float(history_vgg16_a.history["loss"][best_epoch_a - 1])
train_acc_best_a = float(history_vgg16_a.history["accuracy"][best_epoch_a - 1])
gap_a = train_acc_best_a - eval_a["validation_accuracy"]

print(f"\nResults for {EXP_A_ID}:")
print(f"  Actual Epochs Trained:    {len(history_vgg16_a.history['loss'])}")
print(f"  Best Epoch:               {best_epoch_a}")
print(f"  Best Val Loss:            {best_val_loss_a:.4f}")
print(f"  Validation Accuracy:      {eval_a['validation_accuracy']:.4f}")
print(f"  Validation Macro F1:      {eval_a['validation_macro_f1']:.4f}")
print(f"  Generalization Gap:       {gap_a:.4f}")
print(f"  Training Duration:        {duration_a:.2f}s")

# Save experiment config
config_a = {
    "experiment_id": EXP_A_ID,
    "description": "Frozen VGG16 backbone + GAP + Dense(256) + Dropout(0.30)",
    "backbone_trainable": False,
    "dropout": 0.30,
    "optimizer": "Adam",
    "learning_rate": LR_A,
    "max_epochs": MAX_EPOCHS,
    "actual_epochs": len(history_vgg16_a.history["loss"]),
    "best_epoch": best_epoch_a,
    "best_val_loss": best_val_loss_a,
    "validation_accuracy": eval_a["validation_accuracy"],
    "validation_macro_f1": eval_a["validation_macro_f1"],
    "generalization_gap": gap_a,
    "training_duration_seconds": duration_a,
    "total_parameters": int(params_a_total),
    "trainable_parameters": int(params_a_trainable),
    "seed": SEED
}
with open(os.path.join(CONFIGS_DIR, "vgg16_a_config.json"), "w") as f:
    json.dump(config_a, f, indent=4)

### 3.4 — Experiment VGG16-B: Controlled Regularization (Dropout 0.50)

**Hypothesis:** Increasing dropout from 0.30 to 0.50 in the classifier head will reduce co-adaptation between units in the 256-dimensional hidden layer, narrowing the generalization gap between training and validation accuracy.

**Controlled Variable:** Only `dropout_rate` is modified (0.30 -> 0.50). All other hyperparameters, backbone state, and data order remain identical.

In [ ]:
globals().pop('model_vgg16_a', None)
globals().pop('base_vgg16_a', None)
tf.keras.backend.clear_session()
gc.collect()
EXP_B_ID = "VGG16_B"
print(f"\n{'='*70}\nStarting Experiment {EXP_B_ID}\n{'='*70}")

# Reset seeds for reproducibility
tf.keras.utils.set_random_seed(SEED)

# Instantiate fresh base model
base_vgg16_b = tf.keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE
)
base_vgg16_b.trainable = False

# Build model with Dropout 0.50
model_vgg16_b = build_vgg16_model(
    base_model=base_vgg16_b,
    augmentation_layer=make_augmentation(SEED),
    dropout_rate=0.50,
    dense_units=256,
    num_classes=NUM_CLASSES,
    model_name=EXP_B_ID
)

# Compile model
LR_B = 0.001
model_vgg16_b.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_B),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

# Count parameters
params_b_total = model_vgg16_b.count_params()
params_b_trainable = sum(tf.keras.backend.count_params(w) for w in model_vgg16_b.trainable_weights)
params_b_nontrainable = sum(tf.keras.backend.count_params(w) for w in model_vgg16_b.non_trainable_weights)

print(f"Model: {EXP_B_ID}")
print(f"  Total parameters:         {params_b_total:,}")
print(f"  Trainable parameters:     {params_b_trainable:,}")
print(f"  Non-trainable parameters: {params_b_nontrainable:,}")

# Setup checkpoint and early stopping
checkpoint_b_path = os.path.join(CHECKPOINTS_DIR, "vgg16_b_checkpoint.keras")
callbacks_b = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_b_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

# Train model
start_time_b = time.time()
history_vgg16_b = model_vgg16_b.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks_b,
    verbose=1
)
duration_b = time.time() - start_time_b

# Save history
hist_b_df = pd.DataFrame(history_vgg16_b.history)
hist_b_path = os.path.join(HISTORIES_DIR, "vgg16_b_history.csv")
hist_b_df.to_csv(hist_b_path, index=False)
print(f"History saved to: {hist_b_path}")

# Plot learning curves
plot_b_path = os.path.join(PLOTS_DIR, "vgg16_b_learning_curves.png")
best_epoch_b, best_val_loss_b, best_val_acc_b = plot_learning_curves(history_vgg16_b, EXP_B_ID, plot_b_path)

# Evaluate on validation set
eval_b = evaluate_validation_performance(model_vgg16_b, val_ds, val_df, EXP_B_ID)
train_loss_best_b = float(history_vgg16_b.history["loss"][best_epoch_b - 1])
train_acc_best_b = float(history_vgg16_b.history["accuracy"][best_epoch_b - 1])
gap_b = train_acc_best_b - eval_b["validation_accuracy"]

print(f"\nResults for {EXP_B_ID}:")
print(f"  Actual Epochs Trained:    {len(history_vgg16_b.history['loss'])}")
print(f"  Best Epoch:               {best_epoch_b}")
print(f"  Best Val Loss:            {best_val_loss_b:.4f}")
print(f"  Validation Accuracy:      {eval_b['validation_accuracy']:.4f}")
print(f"  Validation Macro F1:      {eval_b['validation_macro_f1']:.4f}")
print(f"  Generalization Gap:       {gap_b:.4f}")
print(f"  Training Duration:        {duration_b:.2f}s")

# Save experiment config
config_b = {
    "experiment_id": EXP_B_ID,
    "description": "Frozen VGG16 backbone + GAP + Dense(256) + Dropout(0.50)",
    "backbone_trainable": False,
    "dropout": 0.50,
    "optimizer": "Adam",
    "learning_rate": LR_B,
    "max_epochs": MAX_EPOCHS,
    "actual_epochs": len(history_vgg16_b.history["loss"]),
    "best_epoch": best_epoch_b,
    "best_val_loss": best_val_loss_b,
    "validation_accuracy": eval_b["validation_accuracy"],
    "validation_macro_f1": eval_b["validation_macro_f1"],
    "generalization_gap": gap_b,
    "training_duration_seconds": duration_b,
    "total_parameters": int(params_b_total),
    "trainable_parameters": int(params_b_trainable),
    "seed": SEED
}
with open(os.path.join(CONFIGS_DIR, "vgg16_b_config.json"), "w") as f:
    json.dump(config_b, f, indent=4)

### 3.5 — Experiment VGG16-C: Scientifically Controlled Fine-Tuning of Upper Layers

**Hypothesis:** By unfreezing the final convolutional block of VGG16 (`block5_conv1`, `block5_conv2`, `block5_conv3`) and training with a significantly smaller learning rate ($\eta=10^{-5}$), the high-level convolutional filters adapt to brain tumour MRI features without destroying the low-level edge/texture representations in Blocks 1–4.

**Strict Fine-Tuning Rules:**
1. **Selective Unfreezing:** Only `block5_conv1`, `block5_conv2`, and `block5_conv3` are set to `trainable=True`. Blocks 1 through 4 remain strictly frozen.
2. **Reduced Learning Rate:** Learning rate is reduced by 100x from $10^{-3}$ to $10^{-5}$ (`0.00001`) to prevent catastrophic forgetting of ImageNet features.
3. **Explicit Recompilation:** Model must be recompiled immediately after modifying layer `trainable` flags.
4. **Fair Budget:** Exactly the same budget (max 20 epochs, patience 4) to ensure fairness with VGG16-A and VGG16-B.

In [ ]:
globals().pop('model_vgg16_b', None)
globals().pop('base_vgg16_b', None)
tf.keras.backend.clear_session()
gc.collect()
EXP_C_ID = "VGG16_C"
print(f"\n{'='*70}\nStarting Experiment {EXP_C_ID}\n{'='*70}")

# Reset seeds for reproducibility
tf.keras.utils.set_random_seed(SEED)

# Select best dropout from frozen experiments A and B based on val_loss
selected_dropout = 0.30 if best_val_loss_a <= best_val_loss_b else 0.50
print(f"Using best dropout from frozen stage: {selected_dropout}")

# Instantiate fresh VGG16 base model
base_vgg16_c = tf.keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=INPUT_SHAPE
)

# Selectively unfreeze only Block 5 convolutional layers
base_vgg16_c.trainable = True
fine_tune_layer_names = ["block5_conv1", "block5_conv2", "block5_conv3"]

for layer in base_vgg16_c.layers:
    if layer.name in fine_tune_layer_names:
        layer.trainable = True
    else:
        layer.trainable = False

# Build model with selective fine-tuning
model_vgg16_c = build_vgg16_model(
    base_model=base_vgg16_c,
    augmentation_layer=make_augmentation(SEED),
    dropout_rate=selected_dropout,
    dense_units=256,
    num_classes=NUM_CLASSES,
    model_name=EXP_C_ID
)

# Recompile with lower learning rate for fine-tuning
LR_C = 1e-5
model_vgg16_c.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_C),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

# Print layer status verification
print("\nVGG16-C Backbone Layer Trainability Status:")
frozen_count = 0
trainable_count = 0
for i, layer in enumerate(base_vgg16_c.layers):
    status = "TRAINABLE" if layer.trainable else "FROZEN"
    if layer.trainable:
        trainable_count += 1
    else:
        frozen_count += 1
    print(f"  [{i:02d}] {layer.name:<18} -> {status}")

params_c_total = model_vgg16_c.count_params()
params_c_trainable = sum(tf.keras.backend.count_params(w) for w in model_vgg16_c.trainable_weights)
params_c_nontrainable = sum(tf.keras.backend.count_params(w) for w in model_vgg16_c.non_trainable_weights)

print(f"\nSummary of Layers in Backbone:")
print(f"  Total backbone layers:    {len(base_vgg16_c.layers)}")
print(f"  Frozen layers:            {frozen_count}")
print(f"  Trainable layers:         {trainable_count} ({fine_tune_layer_names})")
print(f"\nParameters for {EXP_C_ID}:")
print(f"  Total parameters:         {params_c_total:,}")
print(f"  Trainable parameters:     {params_c_trainable:,}")
print(f"  Non-trainable parameters: {params_c_nontrainable:,}")

# Setup checkpoint and early stopping
checkpoint_c_path = os.path.join(CHECKPOINTS_DIR, "vgg16_c_checkpoint.keras")
callbacks_c = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_c_path,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
]

# Train fine-tuned model
start_time_c = time.time()
history_vgg16_c = model_vgg16_c.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks_c,
    verbose=1
)
duration_c = time.time() - start_time_c

# Save history
hist_c_df = pd.DataFrame(history_vgg16_c.history)
hist_c_path = os.path.join(HISTORIES_DIR, "vgg16_c_history.csv")
hist_c_df.to_csv(hist_c_path, index=False)
print(f"History saved to: {hist_c_path}")

# Plot learning curves
plot_c_path = os.path.join(PLOTS_DIR, "vgg16_c_learning_curves.png")
best_epoch_c, best_val_loss_c, best_val_acc_c = plot_learning_curves(history_vgg16_c, EXP_C_ID, plot_c_path)

# Evaluate on validation set
eval_c = evaluate_validation_performance(model_vgg16_c, val_ds, val_df, EXP_C_ID)
train_loss_best_c = float(history_vgg16_c.history["loss"][best_epoch_c - 1])
train_acc_best_c = float(history_vgg16_c.history["accuracy"][best_epoch_c - 1])
gap_c = train_acc_best_c - eval_c["validation_accuracy"]

print(f"\nResults for {EXP_C_ID}:")
print(f"  Actual Epochs Trained:    {len(history_vgg16_c.history['loss'])}")
print(f"  Best Epoch:               {best_epoch_c}")
print(f"  Best Val Loss:            {best_val_loss_c:.4f}")
print(f"  Validation Accuracy:      {eval_c['validation_accuracy']:.4f}")
print(f"  Validation Macro F1:      {eval_c['validation_macro_f1']:.4f}")
print(f"  Generalization Gap:       {gap_c:.4f}")
print(f"  Training Duration:        {duration_c:.2f}s")

# Save experiment config
config_c = {
    "experiment_id": EXP_C_ID,
    "description": "Fine-tuned VGG16 (Block 5 unfrozen) + GAP + Dense(256) + Dropout(selected)",
    "backbone_trainable": True,
    "fine_tuned_layers": fine_tune_layer_names,
    "dropout": selected_dropout,
    "optimizer": "Adam",
    "learning_rate": LR_C,
    "max_epochs": MAX_EPOCHS,
    "actual_epochs": len(history_vgg16_c.history["loss"]),
    "best_epoch": best_epoch_c,
    "best_val_loss": best_val_loss_c,
    "validation_accuracy": eval_c["validation_accuracy"],
    "validation_macro_f1": eval_c["validation_macro_f1"],
    "generalization_gap": gap_c,
    "training_duration_seconds": duration_c,
    "total_parameters": int(params_c_total),
    "trainable_parameters": int(params_c_trainable),
    "seed": SEED
}
with open(os.path.join(CONFIGS_DIR, "vgg16_c_config.json"), "w") as f:
    json.dump(config_c, f, indent=4)

### 3.6 — Phase 3 Experiment Comparison & Summary Table

We compile all three controlled experiments into a consolidated summary table adhering strictly to the group's comparison schema. Notice that all `test_*` fields remain empty/null to guarantee zero test leakage.

In [ ]:
summary_data = [
    {
        "model": "VGG16",
        "experiment_id": EXP_A_ID,
        "description": "Frozen backbone + GAP + Dense(256) + Dropout(0.30)",
        "dropout": 0.30,
        "fine_tuning": False,
        "image_size": f"{IMAGE_HEIGHT}x{IMAGE_WIDTH}",
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": LR_A,
        "max_epochs": MAX_EPOCHS,
        "actual_epochs": len(history_vgg16_a.history["loss"]),
        "best_epoch": best_epoch_a,
        "train_loss_at_best_epoch": train_loss_best_a,
        "best_val_loss": best_val_loss_a,
        "train_accuracy_at_best_epoch": train_acc_best_a,
        "best_val_accuracy": best_val_acc_a,
        "validation_accuracy": eval_a["validation_accuracy"],
        "validation_macro_precision": eval_a["validation_macro_precision"],
        "validation_macro_recall": eval_a["validation_macro_recall"],
        "validation_macro_f1": eval_a["validation_macro_f1"],
        "accuracy_generalization_gap": gap_a,
        "training_time_seconds": duration_a,
        "total_parameters": int(params_a_total),
        "trainable_parameters": int(params_a_trainable),
        "seed": SEED,
        "test_accuracy": np.nan,
        "test_macro_precision": np.nan,
        "test_macro_recall": np.nan,
        "test_macro_f1": np.nan
    },
    {
        "model": "VGG16",
        "experiment_id": EXP_B_ID,
        "description": "Frozen backbone + GAP + Dense(256) + Dropout(0.50)",
        "dropout": 0.50,
        "fine_tuning": False,
        "image_size": f"{IMAGE_HEIGHT}x{IMAGE_WIDTH}",
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": LR_B,
        "max_epochs": MAX_EPOCHS,
        "actual_epochs": len(history_vgg16_b.history["loss"]),
        "best_epoch": best_epoch_b,
        "train_loss_at_best_epoch": train_loss_best_b,
        "best_val_loss": best_val_loss_b,
        "train_accuracy_at_best_epoch": train_acc_best_b,
        "best_val_accuracy": best_val_acc_b,
        "validation_accuracy": eval_b["validation_accuracy"],
        "validation_macro_precision": eval_b["validation_macro_precision"],
        "validation_macro_recall": eval_b["validation_macro_recall"],
        "validation_macro_f1": eval_b["validation_macro_f1"],
        "accuracy_generalization_gap": gap_b,
        "training_time_seconds": duration_b,
        "total_parameters": int(params_b_total),
        "trainable_parameters": int(params_b_trainable),
        "seed": SEED,
        "test_accuracy": np.nan,
        "test_macro_precision": np.nan,
        "test_macro_recall": np.nan,
        "test_macro_f1": np.nan
    },
    {
        "model": "VGG16",
        "experiment_id": EXP_C_ID,
        "description": f"Fine-tuned Block 5 + GAP + Dense(256) + Dropout({selected_dropout})",
        "dropout": selected_dropout,
        "fine_tuning": True,
        "image_size": f"{IMAGE_HEIGHT}x{IMAGE_WIDTH}",
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": LR_C,
        "max_epochs": MAX_EPOCHS,
        "actual_epochs": len(history_vgg16_c.history["loss"]),
        "best_epoch": best_epoch_c,
        "train_loss_at_best_epoch": train_loss_best_c,
        "best_val_loss": best_val_loss_c,
        "train_accuracy_at_best_epoch": train_acc_best_c,
        "best_val_accuracy": best_val_acc_c,
        "validation_accuracy": eval_c["validation_accuracy"],
        "validation_macro_precision": eval_c["validation_macro_precision"],
        "validation_macro_recall": eval_c["validation_macro_recall"],
        "validation_macro_f1": eval_c["validation_macro_f1"],
        "accuracy_generalization_gap": gap_c,
        "training_time_seconds": duration_c,
        "total_parameters": int(params_c_total),
        "trainable_parameters": int(params_c_trainable),
        "seed": SEED,
        "test_accuracy": np.nan,
        "test_macro_precision": np.nan,
        "test_macro_recall": np.nan,
        "test_macro_f1": np.nan
    }
]

summary_df = pd.DataFrame(summary_data)
summary_csv_path = os.path.join(PHASE3_DIR, "vgg16_phase3_experiment_summary.csv")
summary_df.to_csv(summary_csv_path, index=False)
print(f"Phase 3 Experiment Summary saved to: {summary_csv_path}")

# Display formatted summary table
display_cols = [
    "experiment_id", "description", "dropout", "learning_rate", "best_epoch",
    "best_val_loss", "validation_accuracy", "validation_macro_f1",
    "accuracy_generalization_gap", "training_time_seconds", "trainable_parameters"
]
summary_df[display_cols]

### 3.7 — Phase 3 Comparative Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
exp_names = [r["experiment_id"] for r in summary_data]
colors = ["#4c72b0", "#dd8452", "#55a868"]

# 1. Validation Loss (Lower is better — Primary Criterion)
val_losses = [r["best_val_loss"] for r in summary_data]
bars1 = axes[0, 0].bar(exp_names, val_losses, color=colors, width=0.5, edgecolor='black', linewidth=1)
axes[0, 0].set_title("Best Validation Loss (Primary Selection Criterion)", fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel("Validation Loss (Lower is Better)", fontsize=11)
axes[0, 0].grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars1:
    yval = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')

# 2. Validation Macro F1 (Higher is better — Secondary Criterion)
macro_f1s = [r["validation_macro_f1"] for r in summary_data]
bars2 = axes[0, 1].bar(exp_names, macro_f1s, color=colors, width=0.5, edgecolor='black', linewidth=1)
axes[0, 1].set_title("Validation Macro F1 (Secondary Selection Criterion)", fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel("Macro F1 Score (Higher is Better)", fontsize=11)
axes[0, 1].set_ylim([min(macro_f1s) - 0.05, 1.0])
axes[0, 1].grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars2:
    yval = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')

# 3. Validation Accuracy
val_accs = [r["validation_accuracy"] for r in summary_data]
bars3 = axes[1, 0].bar(exp_names, val_accs, color=colors, width=0.5, edgecolor='black', linewidth=1)
axes[1, 0].set_title("Validation Accuracy across Experiments", fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel("Accuracy", fontsize=11)
axes[1, 0].set_ylim([min(val_accs) - 0.05, 1.0])
axes[1, 0].grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars3:
    yval = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2.0, yval + 0.005, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')

# 4. Generalization Gap (Train Acc - Val Acc)
gaps = [r["accuracy_generalization_gap"] for r in summary_data]
bars4 = axes[1, 1].bar(exp_names, gaps, color=colors, width=0.5, edgecolor='black', linewidth=1)
axes[1, 1].axhline(0, color='gray', linestyle='--')
axes[1, 1].set_title("Accuracy Generalization Gap (Train Acc - Val Acc)", fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel("Gap (Smaller indicates less overfitting)", fontsize=11)
axes[1, 1].grid(axis='y', linestyle='--', alpha=0.6)
for bar in bars4:
    yval = bar.get_height()
    pos = yval + 0.002 if yval >= 0 else yval - 0.008
    axes[1, 1].text(bar.get_x() + bar.get_width()/2.0, pos, f"{yval:.4f}", ha='center', va='bottom', fontweight='bold')

plt.suptitle("VGG16 Phase 3: Controlled Validation Experiments Comparison", fontsize=15, y=1.02)
plt.tight_layout()
comparison_plot_path = os.path.join(PLOTS_DIR, "vgg16_phase3_comparison_plots.png")
plt.savefig(comparison_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Comparison plot saved to: {comparison_plot_path}")

### 3.8 — Automated Model Selection Based on Validation Evidence

Per the group experimental contract:
- **Primary Selection Metric:** Minimum Validation Loss (`val_loss`).
- **Secondary Supporting Evidence:** Validation Macro F1.
- **Test Set Independence:** The selection is frozen **BEFORE** opening the test set.
- The winning checkpoint is copied to `selected_vgg16_best.keras`.

In [ ]:
# Programmatic selection using primary criterion (lowest val_loss)
sorted_experiments = sorted(summary_data, key=lambda x: (x["best_val_loss"], -x["validation_macro_f1"]))
selected_exp = sorted_experiments[0]
selected_exp_id = selected_exp["experiment_id"]

print(f"\n{'*'*70}")
print(f"MODEL SELECTION RESULT: {selected_exp_id}")
print(f"{'*'*70}")
print(f"Selection Rationale:")
print(f"  Primary Criterion (Minimum Val Loss):  {selected_exp['best_val_loss']:.4f}")
print(f"  Secondary Evidence (Validation Macro F1): {selected_exp['validation_macro_f1']:.4f}")
print(f"  Validation Accuracy:                   {selected_exp['validation_accuracy']:.4f}")
print(f"  Selected Model Description:            {selected_exp['description']}")

# Map ID to checkpoint file
checkpoint_map = {
    "VGG16_A": checkpoint_a_path,
    "VGG16_B": checkpoint_b_path,
    "VGG16_C": checkpoint_c_path
}
winning_checkpoint = checkpoint_map[selected_exp_id]
selected_model_path = os.path.join(PHASE3_DIR, "selected_vgg16_best.keras")

# Copy winning model checkpoint to frozen destination
shutil.copyfile(winning_checkpoint, selected_model_path)
print(f"\nCopied winning checkpoint ({winning_checkpoint}) to:")
print(f"  {selected_model_path}")
print(f"  File size: {os.path.getsize(selected_model_path) / (1024*1024):.2f} MB")

# Save selected configuration JSON matching Member 1 format
selected_configuration = {
    "selected_experiment": selected_exp_id,
    "selection_primary_metric": "minimum validation loss",
    "selection_secondary_metric": "validation macro F1",
    "test_set_used_for_selection": False,
    "experiment_results": selected_exp
}
selected_config_path = os.path.join(PHASE3_DIR, "selected_vgg16_configuration.json")
with open(selected_config_path, "w") as f:
    json.dump(selected_configuration, f, indent=4)
print(f"Selected configuration saved to: {selected_config_path}")

### 3.9 — Validation Classification Report and Confusion Matrix for Selected Model

We generate the detailed classification report and confusion matrix for the selected model on the validation dataset to diagnose class-specific strengths and weaknesses before any test evaluation.

In [ ]:
# Load predictions for the selected experiment
eval_map = {
    "VGG16_A": eval_a,
    "VGG16_B": eval_b,
    "VGG16_C": eval_c
}
selected_eval = eval_map[selected_exp_id]
y_val_true = selected_eval["y_true"]
y_val_pred = selected_eval["y_pred"]

# Classification Report
val_report_dict = classification_report(y_val_true, y_val_pred, target_names=CLASS_NAMES, output_dict=True)
val_report_df = pd.DataFrame(val_report_dict).transpose()
val_report_csv_path = os.path.join(PHASE3_DIR, "selected_vgg16_validation_classification_report.csv")
val_report_df.to_csv(val_report_csv_path)
print(f"Selected model validation classification report saved to: {val_report_csv_path}")
print("\nClassification Report (Validation Set):")
print(classification_report(y_val_true, y_val_pred, target_names=CLASS_NAMES, digits=4))

# Confusion Matrices (Raw and Normalized)
cm_raw = confusion_matrix(y_val_true, y_val_pred)
cm_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Raw Confusion Matrix
sns.heatmap(cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1, cbar=False)
ax1.set_title(f'{selected_exp_id} — Validation Confusion Matrix (Raw Counts)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Predicted Class', fontsize=11)
ax1.set_ylabel('True Class', fontsize=11)

# Normalized Confusion Matrix
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2, cbar=True)
ax2.set_title(f'{selected_exp_id} — Validation Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Predicted Class', fontsize=11)
ax2.set_ylabel('True Class', fontsize=11)

plt.suptitle(f'Validation Confusion Matrices for Selected Model ({selected_exp_id})', fontsize=14, y=1.02)
plt.tight_layout()
cm_plot_path = os.path.join(PLOTS_DIR, "selected_vgg16_validation_confusion_matrix.png")
plt.savefig(cm_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Validation confusion matrix saved to: {cm_plot_path}")

---
## Phase 3 — Complete

### Summary of Completed Work
1. **Controlled Experiments:** Three transfer-learning experiments (VGG16-A, VGG16-B, VGG16-C) were conducted with rigorous controls (same data pipeline, same seed 42, same batch size 16, same maximum 20 epoch budget).
2. **Strict Validation-Only Selection:** The test set was **NOT** used. Selection was governed strictly by minimum validation loss (`val_loss`) and supported by validation Macro F1.
3. **Fine-Tuning Control:** VGG16-C unfreezes only Block 5 conv layers (`block5_conv1`, `block5_conv2`, `block5_conv3`), keeping Blocks 1–4 frozen, and trained with a reduced learning rate of $10^{-5}$.
4. **Artifact Preservation:** All histories, checkpoint files, validation predictions, configs, comparison plots, and classification reports were saved to `results/vgg16/phase3_experiments/`.
5. **Winning Model Frozen:** The selected model checkpoint was copied to `selected_vgg16_best.keras`.

**STOP — Do not evaluate on the held-out test set until explicitly instructed.**

---
## Phase 4 — First and Final Held-Out Test Evaluation of Frozen VGG16

**Goal:** Perform the **first and final evaluation** of the selected, frozen VGG16 model on the completely held-out test set (`test.csv`, 1,311 images).

### Strict Post-Test Rules
1. **Zero Retraining or Re-tuning:** The model architecture, weights, hyperparameters, and decision thresholds are **frozen**. No further modifications can be made regardless of test set performance.
2. **Data Leakage Prohibition:** The test set was kept completely untouched during Phases 1, 2, and 3. This single evaluation represents authentic generalization to unseen data.
3. **Evidence-Based Error Analysis:** Misclassifications are rigorously cataloged and analyzed without fabricating post-hoc explanations.
4. **Standardized Inference Timing:** Benchmarked using identical protocol across all group members: already-decoded tensor batch of 16 images, 5 warmup runs, 50 timed forward passes synchronized with `.numpy()`.

### 4.1 — Setup Phase 4 Output Directories

In [ ]:
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score, confusion_matrix,
    classification_report, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize

# Define Phase 4 directory paths
PHASE4_DIR = os.path.join(RESULTS_DIR, "phase4_final_test")
PHASE4_ERROR_DIR = os.path.join(PHASE4_DIR, "error_analysis")
PHASE4_PLOTS_DIR = os.path.join(PHASE4_DIR, "plots")

for d in [PHASE4_DIR, PHASE4_ERROR_DIR, PHASE4_PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Phase 4 directories created:")
print(f"  Phase 4 Root:     {PHASE4_DIR}")
print(f"  Error Analysis:   {PHASE4_ERROR_DIR}")
print(f"  Plots:            {PHASE4_PLOTS_DIR}")

### 4.2 — Load the Selected Frozen VGG16 Model

We load the winning checkpoint frozen in Phase 3 (`selected_vgg16_best.keras`). We also load `selected_vgg16_configuration.json` to verify the selected experiment ID and retrieve its training history.

In [ ]:
# Load selected configuration from Phase 3
selected_config_path = os.path.join(PHASE3_DIR, "selected_vgg16_configuration.json")
with open(selected_config_path, "r") as f:
    selected_config_meta = json.load(f)

selected_exp_id = selected_config_meta["selected_experiment"]
print(f"Selected Winning Experiment: {selected_exp_id}")
print(f"Selection Primary Metric:   {selected_config_meta['selection_primary_metric']}")

# Load the frozen winning model
frozen_model_path = os.path.join(PHASE3_DIR, "selected_vgg16_best.keras")
final_vgg16_model = tf.keras.models.load_model(frozen_model_path)
print(f"Successfully loaded frozen model from: {frozen_model_path}")

# Verify parameters
total_params = final_vgg16_model.count_params()
trainable_params = sum(tf.keras.backend.count_params(w) for w in final_vgg16_model.trainable_weights)
non_trainable_params = sum(tf.keras.backend.count_params(w) for w in final_vgg16_model.non_trainable_weights)
model_size_mb = os.path.getsize(frozen_model_path) / (1024 * 1024)

print(f"\nFinal VGG16 Model Architecture Summary:")
print(f"  Total Parameters:         {total_params:,}")
print(f"  Trainable Parameters:     {trainable_params:,}")
print(f"  Non-Trainable Parameters: {non_trainable_params:,}")
print(f"  Model Checkpoint Size:    {model_size_mb:.2f} MB")

# Copy and preserve the final training history
src_history_path = os.path.join(HISTORIES_DIR, f"{selected_exp_id.lower()}_history.csv")
final_history_path = os.path.join(PHASE4_DIR, "vgg16_final_training_history.csv")
shutil.copyfile(src_history_path, final_history_path)
final_history_df = pd.read_csv(final_history_path)
print(f"Final training history preserved at: {final_history_path}")

### 4.3 — Build the Held-Out Test Dataset (`tf.data`)

We now build the test dataset from `splits/test.csv` (1,311 images).

- `shuffle = False` to preserve strict 1-to-1 correspondence with the test manifest.
- `tf.data` returns resized float32 RGB images in the raw `[0,255]` range.
- **No augmentation** is applied by `tf.data`.
- The frozen saved model applies `vgg16_preprocess_input` internally exactly once.
- Because inference uses `training=False`, the model's augmentation layers are inactive.


In [ ]:
# Build test dataset using the verified pipeline
test_ds = build_dataset(test_df, is_training=False)
num_test_batches = tf.data.experimental.cardinality(test_ds).numpy()
print(f"Held-Out Test Dataset Built:")
print(f"  Total Samples: {len(test_df)} images")
print(f"  Batch Size:    {BATCH_SIZE}")
print(f"  Total Batches: {num_test_batches} (81 batches of 16 + 1 batch of 15)")

# Verify unshuffled order by comparing first batch labels with test_df
expected_first_test_labels = [CLASS_TO_INDEX[l] for l in test_df["label"].values[:BATCH_SIZE]]
for _, test_labels_batch in test_ds.take(1):
    actual_first_test_labels = test_labels_batch.numpy().tolist()
assert expected_first_test_labels == actual_first_test_labels, "Test dataset order mismatch!"
print("Test dataset unshuffled ordering verified.")

### 4.4 — Execute Final Test Evaluation & Prediction Inference

In [ ]:
print("\n" + "="*70)
print("EXECUTING FINAL TEST EVALUATION (HELD-OUT TEST SET)")
print("="*70)

# 1. Model Evaluate (Loss and Accuracy)
eval_results = final_vgg16_model.evaluate(test_ds, verbose=1)
test_loss = float(eval_results[0])
test_accuracy = float(eval_results[1])

# 2. Predict Probabilities across the complete test set
t0_pred = time.time()
test_probabilities = final_vgg16_model.predict(test_ds, verbose=1)
full_test_prediction_time = time.time() - t0_pred

# Argmax class prediction & confidence
test_predicted_indices = np.argmax(test_probabilities, axis=1)
test_predicted_labels = [CLASS_NAMES[i] for i in test_predicted_indices]
test_confidences = np.max(test_probabilities, axis=1)

# True labels from test.csv
test_true_indices = np.array([CLASS_TO_INDEX[l] for l in test_df["label"].values])
test_true_labels = test_df["label"].values
correct_mask = (test_true_indices == test_predicted_indices)

# 3. Compute Scikit-Learn Macro & Weighted Metrics
macro_prec, macro_rec, macro_f1, _ = precision_recall_fscore_support(
    test_true_indices, test_predicted_indices, average='macro', zero_division=0
)
weighted_prec, weighted_rec, weighted_f1, _ = precision_recall_fscore_support(
    test_true_indices, test_predicted_indices, average='weighted', zero_division=0
)

print(f"\n{'='*70}")
print(f"FINAL HELD-OUT TEST RESULTS:")
print(f"{'='*70}")
print(f"  Test Loss:              {test_loss:.6f}")
print(f"  Test Accuracy:          {test_accuracy:.6f} ({np.sum(correct_mask)}/{len(test_df)} correct)")
print(f"  Macro Precision:        {macro_prec:.6f}")
print(f"  Macro Recall:           {macro_rec:.6f}")
print(f"  Macro F1-Score:         {macro_f1:.6f}")
print(f"  Weighted Precision:     {weighted_prec:.6f}")
print(f"  Weighted Recall:        {weighted_rec:.6f}")
print(f"  Weighted F1-Score:      {weighted_f1:.6f}")
print(f"  Total Prediction Time:  {full_test_prediction_time:.2f}s")

### 4.5 — Save Detailed Test Predictions Manifest

Saves image-level predictions with probabilities, predicted vs true labels, and confidence to `vgg16_test_predictions.csv`.

In [ ]:
test_predictions_df = pd.DataFrame({
    "filepath": test_df["filepath"].values,
    "true_index": test_true_indices,
    "true_label": test_true_labels,
    "predicted_index": test_predicted_indices,
    "predicted_label": test_predicted_labels,
    "correct": correct_mask,
    "prob_glioma": test_probabilities[:, 0],
    "prob_meningioma": test_probabilities[:, 1],
    "prob_notumor": test_probabilities[:, 2],
    "prob_pituitary": test_probabilities[:, 3],
    "prediction_confidence": test_confidences
})

test_pred_csv_path = os.path.join(PHASE4_DIR, "vgg16_test_predictions.csv")
test_predictions_df.to_csv(test_pred_csv_path, index=False)
print(f"Test predictions manifest saved to: {test_pred_csv_path}")
test_predictions_df.head(10)

### 4.6 — Classification Report and Per-Class Performance Summary

In [ ]:
# 1. Classification Report
clf_report_dict = classification_report(
    test_true_indices, test_predicted_indices, target_names=CLASS_NAMES, output_dict=True, digits=6
)
clf_report_df = pd.DataFrame(clf_report_dict).transpose()
clf_report_csv_path = os.path.join(PHASE4_DIR, "vgg16_test_classification_report.csv")
clf_report_df.to_csv(clf_report_csv_path)
print(f"Classification report CSV saved to: {clf_report_csv_path}\n")
print(classification_report(test_true_indices, test_predicted_indices, target_names=CLASS_NAMES, digits=6))

# 2. Per-Class Summary Table (Class accuracy, support, correct, errors)
per_class_rows = []
y_test_bin = label_binarize(test_true_indices, classes=[0, 1, 2, 3])

for idx, cls_name in enumerate(CLASS_NAMES):
    cls_mask = (test_true_indices == idx)
    support = int(np.sum(cls_mask))
    correct = int(np.sum((test_true_indices == idx) & (test_predicted_indices == idx)))
    errors = support - correct
    class_acc = correct / support if support > 0 else 0.0
    cls_roc_auc = float(roc_auc_score(y_test_bin[:, idx], test_probabilities[:, idx]))

    per_class_rows.append({
        "class": cls_name,
        "support": support,
        "correct": correct,
        "errors": errors,
        "class_accuracy": class_acc,
        "roc_auc": cls_roc_auc
    })

per_class_df = pd.DataFrame(per_class_rows)
per_class_csv_path = os.path.join(PHASE4_DIR, "vgg16_test_per_class_summary.csv")
per_class_df.to_csv(per_class_csv_path, index=False)
print(f"\nPer-class summary saved to: {per_class_csv_path}")
per_class_df

### 4.7 — Confusion Matrix Generation (Raw Counts and Normalized)

In [ ]:
test_cm_raw = confusion_matrix(test_true_indices, test_predicted_indices)
test_cm_norm = test_cm_raw.astype('float') / test_cm_raw.sum(axis=1)[:, np.newaxis]

# 1. Plot and save Raw Confusion Matrix
plt.figure(figsize=(7, 6))
sns.heatmap(test_cm_raw, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=False)
plt.title("VGG16 Final Test — Confusion Matrix (Raw Counts)", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("True Label", fontsize=11)
plt.tight_layout()
raw_cm_path = os.path.join(PHASE4_PLOTS_DIR, "vgg16_test_confusion_matrix.png")
plt.savefig(raw_cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Raw confusion matrix saved to: {raw_cm_path}")

# 2. Plot and save Normalized Confusion Matrix
plt.figure(figsize=(7, 6))
sns.heatmap(test_cm_norm, annot=True, fmt='.2%', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cbar=True)
plt.title("VGG16 Final Test — Confusion Matrix (Normalized)", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Label", fontsize=11)
plt.ylabel("True Label", fontsize=11)
plt.tight_layout()
norm_cm_path = os.path.join(PHASE4_PLOTS_DIR, "vgg16_test_confusion_matrix_normalized.png")
plt.savefig(norm_cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Normalized confusion matrix saved to: {norm_cm_path}")

### 4.8 — One-vs-Rest (OvR) ROC Curves and Area Under Curve (ROC-AUC)

We calculate One-vs-Rest ROC curves and AUC for each of the four tumour classes, as well as Macro and Weighted ROC-AUC across all classes.

In [ ]:
# Macro & Weighted Multi-Class ROC-AUC
roc_auc_macro = float(roc_auc_score(y_test_bin, test_probabilities, multi_class='ovr', average='macro'))
roc_auc_weighted = float(roc_auc_score(y_test_bin, test_probabilities, multi_class='ovr', average='weighted'))

print(f"Macro OvR ROC-AUC:    {roc_auc_macro:.6f}")
print(f"Weighted OvR ROC-AUC: {roc_auc_weighted:.6f}")

# Compute ROC curve and AUC for each class
fpr = dict()
tpr = dict()
roc_auc_per_class = dict()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

plt.figure(figsize=(9, 7))
for i, cls_name in enumerate(CLASS_NAMES):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], test_probabilities[:, i])
    roc_auc_per_class[cls_name] = float(auc(fpr[i], tpr[i]))
    plt.plot(fpr[i], tpr[i], color=colors[i], lw=2,
             label=f'{cls_name} (AUC = {roc_auc_per_class[cls_name]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Chance (AUC = 0.5000)')
plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.02])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
plt.title(f'VGG16 One-vs-Rest ROC Curves (Macro AUC = {roc_auc_macro:.4f})', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', frameon=True, fontsize=10)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

roc_plot_path = os.path.join(PHASE4_PLOTS_DIR, "vgg16_test_roc_curves.png")
plt.savefig(roc_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"ROC curves plot saved to: {roc_plot_path}")

### 4.9 — Final Model Learning Curves

Plots the final training and validation loss and accuracy trajectories of the selected model configuration, marking the best epoch.

In [ ]:
epochs_range = range(1, len(final_history_df) + 1)
best_epoch_num = int(selected_config_meta["experiment_results"]["best_epoch"])

# 1. Final Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, final_history_df["loss"], 'o-', color='#1f77b4', lw=2, label='Training Loss')
plt.plot(epochs_range, final_history_df["val_loss"], 's--', color='#ff7f0e', lw=2, label='Validation Loss')
plt.axvline(best_epoch_num, color='crimson', linestyle=':', label=f'Best Checkpoint (Epoch {best_epoch_num})')
plt.title(f'VGG16 Final Model ({selected_exp_id}) — Training & Validation Loss', fontsize=12, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Loss', fontsize=11)
plt.legend(loc='upper right', frameon=True)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
loss_curve_path = os.path.join(PHASE4_PLOTS_DIR, "vgg16_final_loss_curve.png")
plt.savefig(loss_curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Final loss curve saved to: {loss_curve_path}")

# 2. Final Accuracy Curve
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, final_history_df["accuracy"], 'o-', color='#1f77b4', lw=2, label='Training Accuracy')
plt.plot(epochs_range, final_history_df["val_accuracy"], 's--', color='#2ca02c', lw=2, label='Validation Accuracy')
plt.axvline(best_epoch_num, color='crimson', linestyle=':', label=f'Best Checkpoint (Epoch {best_epoch_num})')
plt.title(f'VGG16 Final Model ({selected_exp_id}) — Training & Validation Accuracy', fontsize=12, fontweight='bold')
plt.xlabel('Epoch', fontsize=11)
plt.ylabel('Accuracy', fontsize=11)
plt.ylim([0.0, 1.02])
plt.legend(loc='lower right', frameon=True)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
acc_curve_path = os.path.join(PHASE4_PLOTS_DIR, "vgg16_final_accuracy_curve.png")
plt.savefig(acc_curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Final accuracy curve saved to: {acc_curve_path}")

### 4.10 — Standardized Model-Only Inference Timing Benchmark

We adhere strictly to the shared timing benchmark protocol:
- Fixed batch size = `16`
- Already-decoded, materialized tensor batch (isolating model forward compute from disk I/O)
- `5` warm-up passes (excluding CUDA initialization / graph compilation overhead)
- `50` timed forward passes with `training=False`
- Output materialized with `.numpy()` for strict hardware synchronization
- Record milliseconds per image and throughput (images/sec)

In [ ]:
# Extract one pre-decoded tensor batch of 16 images
for sample_images, _ in test_ds.take(1):
    timing_batch = tf.identity(sample_images)
    break

WARMUP_RUNS = 5
TIMED_RUNS = 50
TIMED_BATCH_SIZE = timing_batch.shape[0]
TOTAL_TIMED_IMAGES = WARMUP_RUNS * 0 + TIMED_RUNS * TIMED_BATCH_SIZE  # 800 images

print(f"Inference Timing Setup:")
print(f"  Tensor Shape:       {timing_batch.shape}")
print(f"  Warmup Passes:      {WARMUP_RUNS}")
print(f"  Timed Passes:       {TIMED_RUNS}")
print(f"  Total Timed Images: {TOTAL_TIMED_IMAGES}")

# 1. Warm-up passes
for _ in range(WARMUP_RUNS):
    _ = final_vgg16_model(timing_batch, training=False).numpy()

# 2. Timed forward passes
t_start = time.perf_counter()
for _ in range(TIMED_RUNS):
    _ = final_vgg16_model(timing_batch, training=False).numpy()
t_elapsed = time.perf_counter() - t_start

ms_per_image = (t_elapsed / TOTAL_TIMED_IMAGES) * 1000.0
throughput_ips = TOTAL_TIMED_IMAGES / t_elapsed

timing_results = {
    "method": "model-only repeated batch forward pass",
    "batch_size": int(TIMED_BATCH_SIZE),
    "warmup_runs": WARMUP_RUNS,
    "timed_runs": TIMED_RUNS,
    "total_timed_images": TOTAL_TIMED_IMAGES,
    "elapsed_seconds": float(t_elapsed),
    "milliseconds_per_image": float(ms_per_image),
    "images_per_second": float(throughput_ips),
    "gpu": environment_info.get("gpu_name", "Unknown GPU")
}

timing_json_path = os.path.join(PHASE4_DIR, "vgg16_inference_timing.json")
with open(timing_json_path, "w") as f:
    json.dump(timing_results, f, indent=4)

print(f"\nTiming Benchmark Results:")
print(f"  Elapsed Time:            {t_elapsed:.4f} seconds")
print(f"  Latency per Image:       {ms_per_image:.4f} ms/image")
print(f"  Inference Throughput:    {throughput_ips:.2f} images/sec")
print(f"  Hardware Platform:       {timing_results['gpu']}")
print(f"Saved timing benchmark to: {timing_json_path}")

### 4.11 — Comprehensive Error Analysis

We analyze every error made on the held-out test set:
1. Extract and save misclassified images to `vgg16_misclassified_images.csv`.
2. Compute the confusion error-pair frequency table (`vgg16_error_pairs.csv`).
3. Visualize high-confidence misclassifications to diagnose clinical/imaging ambiguities.
4. Record evidence-based findings in `vgg16_evidence_notes.json`.

In [ ]:
# 1. Extract misclassifications
misclassified_df = test_predictions_df[~test_predictions_df["correct"]].copy()
misclassified_df.sort_values(by="prediction_confidence", ascending=False, inplace=True)

misclassified_csv_path = os.path.join(PHASE4_ERROR_DIR, "vgg16_misclassified_images.csv")
misclassified_df.to_csv(misclassified_csv_path, index=False)
print(f"Total Misclassified Test Images: {len(misclassified_df)} out of {len(test_df)} ({len(misclassified_df)/len(test_df):.2%})")
print(f"Saved misclassified images to: {misclassified_csv_path}")

# 2. Error Pairs Frequency Table (true_label -> predicted_label)
error_pairs = (
    misclassified_df.groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)
error_pairs_csv_path = os.path.join(PHASE4_ERROR_DIR, "vgg16_error_pairs.csv")
error_pairs.to_csv(error_pairs_csv_path, index=False)
print(f"\nError Pairs Frequency Table saved to: {error_pairs_csv_path}")
print(error_pairs)

# Most common error pair
if len(error_pairs) > 0:
    top_error = error_pairs.iloc[0]
    most_common_error_str = f"{top_error['true_label']} -> {top_error['predicted_label']} ({int(top_error['count'])} cases)"
else:
    most_common_error_str = 'No classification errors'

# Per-class F1 extremes
per_class_f1s = {cls: clf_report_dict[cls]["f1-score"] for cls in CLASS_NAMES}
lowest_f1_cls = min(per_class_f1s, key=per_class_f1s.get)
highest_f1_cls = max(per_class_f1s, key=per_class_f1s.get)
val_acc = float(selected_config_meta["experiment_results"]["validation_accuracy"])
val_test_acc_gap = val_acc - test_accuracy

# 3. Save Evidence Notes JSON
evidence_notes = {
    "lowest_test_f1_class": lowest_f1_cls,
    "highest_test_f1_class": highest_f1_cls,
    "most_common_error_pair": most_common_error_str,
    "validation_test_accuracy_gap": float(val_test_acc_gap),
    "misclassified_images": int(len(misclassified_df)),
    "dataset_limitation": "Patient-level independence cannot be verified from available identifiers.",
    "leakage_control": "Exact duplicate overlap was audited before frozen splits were created.",
    "interpretation_warning": "Performance on this dataset does not establish performance on unseen patients, scanners, hospitals or clinical populations."
}

evidence_notes_path = os.path.join(PHASE4_DIR, "vgg16_evidence_notes.json")
with open(evidence_notes_path, "w") as f:
    json.dump(evidence_notes, f, indent=4)
print(f"Evidence notes saved to: {evidence_notes_path}")

# 4. Plot High-Confidence Misclassified Examples (Top 8)
num_examples = min(8, len(misclassified_df))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("VGG16 Final Test — Top High-Confidence Misclassifications", fontsize=14, fontweight='bold')

for idx in range(num_examples):
    ax = axes.flat[idx]
    row = misclassified_df.iloc[idx]
    img_path = os.path.join(dataset_path, row["filepath"])
    raw_img = tf.io.read_file(img_path)
    decoded_img = tf.image.decode_jpeg(raw_img, channels=3)
    resized_img = tf.image.resize(decoded_img, (224, 224)).numpy().astype(np.uint8)

    ax.imshow(resized_img)
    ax.set_title(
        f"True: {row['true_label']}\nPred: {row['predicted_label']}\nConf: {row['prediction_confidence']:.2%}",
        fontsize=10, color='darkred'
    )
    ax.axis('off')

plt.tight_layout()
misclf_plot_path = os.path.join(PHASE4_PLOTS_DIR, "vgg16_misclassified_examples.png")
plt.savefig(misclf_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Misclassified examples visualization saved to: {misclf_plot_path}")

### 4.12 — Compile Final Metrics (JSON and Single-Row CSV)

We compile all parameters, test performance metrics, timings, and complexity measurements into `vgg16_final_metrics.json` and `vgg16_final_metrics.csv` conforming to the exact group comparative schema.

In [ ]:
training_time_val = float(selected_config_meta["experiment_results"]["training_time_seconds"])
best_epoch_val = int(selected_config_meta["experiment_results"]["best_epoch"])
actual_epochs_val = int(selected_config_meta["experiment_results"]["actual_epochs"])
best_val_loss_val = float(selected_config_meta["experiment_results"]["best_val_loss"])
best_val_acc_val = float(selected_config_meta["experiment_results"]["best_val_accuracy"])

final_metrics_data = {
    "model": "VGG16",
    "selected_experiment": selected_exp_id,
    "dataset_version": DATASET_VERSION,
    "train_samples": len(train_df),
    "validation_samples": len(val_df),
    "test_samples": len(test_df),
    "image_size": f"{IMAGE_HEIGHT}x{IMAGE_WIDTH}x{CHANNELS}",
    "batch_size": BATCH_SIZE,
    "optimizer": "Adam",
    "learning_rate": float(selected_config_meta["experiment_results"]["learning_rate"]),
    "max_epochs": MAX_EPOCHS,
    "actual_epochs": actual_epochs_val,
    "best_epoch": best_epoch_val,
    "best_val_loss": best_val_loss_val,
    "best_val_accuracy": best_val_acc_val,
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "macro_precision": float(macro_prec),
    "macro_recall": float(macro_rec),
    "macro_f1": float(macro_f1),
    "weighted_precision": float(weighted_prec),
    "weighted_recall": float(weighted_rec),
    "weighted_f1": float(weighted_f1),
    "roc_auc_macro": float(roc_auc_macro),
    "roc_auc_weighted": float(roc_auc_weighted),
    "training_time_seconds": training_time_val,
    "full_test_prediction_seconds": float(full_test_prediction_time),
    "inference_ms_per_image": float(ms_per_image),
    "inference_images_per_second": float(throughput_ips),
    "total_parameters": int(total_params),
    "trainable_parameters": int(trainable_params),
    "model_size_mb": float(model_size_mb),
    "seed": SEED,
    "gpu": timing_results["gpu"],
    "test_used_only_after_model_selection": True,
    "patient_level_independence_verified": False
}

# 1. Save Final Metrics JSON
final_metrics_json_path = os.path.join(PHASE4_DIR, "vgg16_final_metrics.json")
with open(final_metrics_json_path, "w") as f:
    json.dump(final_metrics_data, f, indent=4)
print(f"Final metrics JSON saved to: {final_metrics_json_path}")

# 2. Save Final Metrics Single-Row CSV
final_metrics_df = pd.DataFrame([final_metrics_data])
final_metrics_csv_path = os.path.join(PHASE4_DIR, "vgg16_final_metrics.csv")
final_metrics_df.to_csv(final_metrics_csv_path, index=False)
print(f"Final metrics CSV saved to: {final_metrics_csv_path}")

final_metrics_df.transpose()

---
## Phase 4 — Complete

### Summary of Completed Work
1. **Held-Out Test Set Evaluated:** The frozen winning model from Phase 3 was evaluated on the 1,311 test images.
2. **All Final Metrics Calculated:** Test loss, test accuracy, macro and weighted precision/recall/F1, and multi-class OvR ROC-AUC.
3. **Inference Benchmarking:** Standardized model-only latency and throughput measured with 5 warmup and 50 timed passes.
4. **Error Analysis Completed:** Cataloged misclassified images, error pairs, high-confidence mistakes, and evidence notes.
5. **Artifacts Preserved:** Metrics, predictions, classifications reports, plots, and timing details exported.
6. **Strict Post-Test Freezing:** Zero model modifications made following test evaluation.

**STOP — Do not proceed to Phase 5 until instructed.**

---
## Phase 5 — Model Finalization, Reproducibility, and Handoff

**Goal:** Package all final VGG16 model artifacts, ensure end-to-end reproducibility, generate the standardized single-row comparison record, and finalize the model card for collaborative group handoff.

### Phase 5 Checklist
1. Copy selected best model to canonical location: `models/vgg16/vgg16_final.keras`.
2. Generate SHA-256 checksum for the final model binary.
3. Export standardized single-row comparison CSV (`vgg16_comparison_row.csv`) matching the group schema.
4. Export reproducibility configuration (`vgg16_reproducibility_config.json`) and artifact checksums (`artifact_checksums.json`).
5. Save architecture diagram visualization (`vgg16_architecture.png`).
6. Provide complete model card documentation (`results/vgg16/MODEL_CARD.md`).
7. Verify clean top-to-bottom Colab execution without dead code, hardcoded drive paths, or external dependencies.

### 5.1 — Setup Phase 5 Directory and Copy Final Model Binary

In [ ]:
PHASE5_DIR = os.path.join(RESULTS_DIR, "phase5_finalization")
os.makedirs(PHASE5_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Canonical final model destination
final_model_keras_path = os.path.join(MODELS_DIR, "vgg16_final.keras")
src_selected_model = os.path.join(PHASE3_DIR, "selected_vgg16_best.keras")

# Copy final model to canonical models directory
shutil.copyfile(src_selected_model, final_model_keras_path)
print(f"Final VGG16 model copied to: {final_model_keras_path}")
print(f"Model file size: {os.path.getsize(final_model_keras_path) / (1024 * 1024):.2f} MB")

# Compute SHA-256 for final model
final_model_sha256 = compute_sha256(final_model_keras_path)
print(f"Final model SHA-256: {final_model_sha256}")

### 5.2 — Export Standardized Single-Row Comparison CSV

We construct the single-row comparison CSV adhering strictly to the shared group schema across all four deep learning models (Custom CNN, VGG16, ResNet50, DenseNet121).

In [ ]:
# Load final metrics JSON from Phase 4
final_metrics_json_path = os.path.join(PHASE4_DIR, "vgg16_final_metrics.json")
with open(final_metrics_json_path, "r") as f:
    m = json.load(f)

# Construct comparison row matching exact group schema
comparison_row = {
    "model": "VGG16",
    "experiment_id": m["selected_experiment"],
    "image_size": "224x224x3",
    "batch_size": m["batch_size"],
    "optimizer": m["optimizer"],
    "learning_rate": m["learning_rate"],
    "max_epochs": m["max_epochs"],
    "actual_epochs": m["actual_epochs"],
    "best_epoch": m["best_epoch"],
    "best_val_accuracy": m["best_val_accuracy"],
    "test_accuracy": m["test_accuracy"],
    "macro_precision": m["macro_precision"],
    "macro_recall": m["macro_recall"],
    "macro_f1": m["macro_f1"],
    "roc_auc_macro": m["roc_auc_macro"],
    "training_time_seconds": m["training_time_seconds"],
    "inference_time": m["inference_ms_per_image"],
    "inference_time_unit": "milliseconds_per_image",
    "total_parameters": m["total_parameters"],
    "trainable_parameters": m["trainable_parameters"],
    "model_size_mb": m["model_size_mb"],
    "seed": m["seed"],
    "gpu": m["gpu"],
    "dataset_version": m["dataset_version"],
    "test_samples": m["test_samples"],
    "patient_level_independence_verified": m["patient_level_independence_verified"]
}

comparison_df = pd.DataFrame([comparison_row])
comparison_csv_path = os.path.join(PHASE5_DIR, "vgg16_comparison_row.csv")
comparison_df.to_csv(comparison_csv_path, index=False)
print(f"VGG16 comparison row saved to: {comparison_csv_path}")
comparison_df.transpose()

### 5.3 — Export Reproducibility Configuration and Environment Manifest

In [ ]:
# 1. Save Reproducibility Configuration
reproducibility_config = {
    "project": "Comparative Analysis of Deep Learning Architectures for Multi-Class Brain Tumour MRI Classification",
    "member": 2,
    "architecture": "VGG16",
    "selected_experiment": m["selected_experiment"],
    "dataset": "masoudnickparvar/brain-tumor-mri-dataset/versions/1",
    "dataset_fingerprint": "061cbffb7341abf85e57a4f99de571e8fa4702ec8236c929e35543ab0547217d",
    "split_checksums": computed_checksums,
    "train_samples": len(train_df),
    "validation_samples": len(val_df),
    "test_samples": len(test_df),
    "seed": SEED,
    "image_size": [IMAGE_HEIGHT, IMAGE_WIDTH, CHANNELS],
    "class_mapping": CLASS_TO_INDEX,
    "preprocessing": "keras.applications.vgg16.preprocess_input",
    "test_used_only_after_model_selection": True,
    "post_test_model_tuning_performed": False,
    "patient_level_independence_verified": False
}

repro_config_path = os.path.join(PHASE5_DIR, "vgg16_reproducibility_config.json")
with open(repro_config_path, "w") as f:
    json.dump(reproducibility_config, f, indent=4)
print(f"Reproducibility config saved to: {repro_config_path}")

# 2. Save Phase 5 Environment info copy
env_phase5_path = os.path.join(PHASE5_DIR, "vgg16_environment.json")
with open(env_phase5_path, "w") as f:
    json.dump(environment_info, f, indent=4)
print(f"Environment info saved to: {env_phase5_path}")

### 5.4 — Generate Artifact SHA-256 Checksums

In [ ]:
# Generate model card from actual measured outputs (no hard-coded performance values).
model_card_path = os.path.join(RESULTS_DIR, 'MODEL_CARD.md')
model_card = f"""# VGG16 Model Card

## Project
Comparative Analysis of Deep Learning Architectures for Multi-Class Brain Tumour MRI Classification

## Dataset
- Pinned dataset: `masoudnickparvar/brain-tumor-mri-dataset/versions/1`
- Train / validation / test: {len(train_df)} / {len(val_df)} / {len(test_df)}
- Classes: {', '.join(CLASS_NAMES)}
- Patient-level independence verified: **No**

## Model
- Architecture: VGG16 with ImageNet weights (`include_top=False`)
- Selected experiment: `{m['selected_experiment']}`
- Input: `224x224x3`
- Preprocessing: `vgg16_preprocess_input`
- Optimizer: `{m['optimizer']}`
- Learning rate: `{m['learning_rate']}`
- Batch size: `{m['batch_size']}`
- Maximum epochs: `{m['max_epochs']}`
- Actual epochs: `{m['actual_epochs']}`
- Best epoch: `{m['best_epoch']}`

## Final held-out test results
- Accuracy: `{m['test_accuracy']:.6f}`
- Macro precision: `{m['macro_precision']:.6f}`
- Macro recall: `{m['macro_recall']:.6f}`
- Macro F1: `{m['macro_f1']:.6f}`
- Macro ROC-AUC: `{m['roc_auc_macro']:.6f}`

## Complexity and efficiency
- Total parameters: `{int(m['total_parameters']):,}`
- Trainable parameters: `{int(m['trainable_parameters']):,}`
- Model size: `{m['model_size_mb']:.2f} MB`
- Training time: `{m['training_time_seconds']:.2f} s`
- Inference: `{m['inference_ms_per_image']:.4f} ms/image`
- GPU: `{m['gpu']}`

## Limitations
ImageNet pretraining makes this a comparison of practical training approaches rather than an architecture-only isolation. Patient identifiers are unavailable, so patient-level independence cannot be verified. This is an academic classifier, not a clinical diagnostic system.
"""
with open(model_card_path, 'w', encoding='utf-8') as f:
    f.write(model_card)
print('Model card written from actual metrics:', model_card_path)

In [ ]:
# Calculate checksums for generated deliverables.
# NOTE: We intentionally do NOT hash PROJECT_ROOT/notebooks/02_vgg16.ipynb
# here. In Colab, the notebook open in the UI may be newer than the copy
# cloned from GitHub, so hashing the cloned notebook could misrepresent
# the code that actually produced these results.

artifacts_to_hash = {
    "train.csv": os.path.join(SPLITS_DIR, "train.csv"),
    "val.csv": os.path.join(SPLITS_DIR, "val.csv"),
    "test.csv": os.path.join(SPLITS_DIR, "test.csv"),
    "vgg16_final.keras": final_model_keras_path,
    "vgg16_final_metrics.csv": os.path.join(PHASE4_DIR, "vgg16_final_metrics.csv"),
    "vgg16_test_predictions.csv": os.path.join(PHASE4_DIR, "vgg16_test_predictions.csv"),
    "vgg16_comparison_row.csv": comparison_csv_path,
    "MODEL_CARD.md": os.path.join(RESULTS_DIR, "MODEL_CARD.md"),
}

artifact_checksums = {}

for name, path in artifacts_to_hash.items():
    if os.path.exists(path):
        artifact_checksums[name] = compute_sha256(path)
    else:
        artifact_checksums[name] = "FILE_NOT_FOUND"

checksums_json_path = os.path.join(
    PHASE5_DIR,
    "artifact_checksums.json"
)

with open(checksums_json_path, "w") as f:
    json.dump(artifact_checksums, f, indent=4)

print(f"Artifact checksums saved to: {checksums_json_path}")

for name, checksum in artifact_checksums.items():
    print(f"  {name:<32} -> {checksum}")

missing_artifacts = [
    name for name, checksum in artifact_checksums.items()
    if checksum == "FILE_NOT_FOUND"
]

assert not missing_artifacts, (
    f"Expected final artifacts are missing: {missing_artifacts}"
)

print("\n✓ All expected final artifacts exist and were hashed.")


### 5.5 — Save Final Architecture Visualization

In [ ]:
# Generate Keras model plot or structured layer diagram
arch_plot_path = os.path.join(PHASE5_DIR, "vgg16_architecture.png")
try:
    tf.keras.utils.plot_model(
        final_vgg16_model,
        to_file=arch_plot_path,
        show_shapes=True,
        show_layer_names=True,
        dpi=150
    )
    print(f"Keras model architecture diagram saved to: {arch_plot_path}")
except Exception as e:
    # Fallback to matplotlib diagram if pydot/graphviz is not installed
    fig, ax = plt.subplots(figsize=(10, 8))
    layer_info = []
    for i, layer in enumerate(final_vgg16_model.layers):
        train_tag = "Trainable" if layer.trainable else "Frozen"
        params = layer.count_params()
        layer_info.append(f"{i+1}. {layer.name} ({layer.__class__.__name__})\n   Output: {getattr(layer, 'output_shape', 'unknown')} | {params:,} params [{train_tag}]")
    ax.text(0.05, 0.95, "\n\n".join(layer_info), fontsize=9, va='top', fontfamily='monospace')
    ax.axis('off')
    plt.title(f"VGG16 Model Architecture — {selected_exp_id}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(arch_plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Fallback architecture diagram saved to: {arch_plot_path}")

In [ ]:
# ============================================================
# FINAL LIGHTWEIGHT HANDOFF EXPORT
# ============================================================

HANDOFF_DIR = os.path.join(PHASE5_DIR, "handoff_lightweight")
os.makedirs(HANDOFF_DIR, exist_ok=True)

handoff_sources = {
    "vgg16_final_metrics.json": os.path.join(PHASE4_DIR, "vgg16_final_metrics.json"),
    "vgg16_final_metrics.csv": os.path.join(PHASE4_DIR, "vgg16_final_metrics.csv"),
    "vgg16_test_predictions.csv": os.path.join(PHASE4_DIR, "vgg16_test_predictions.csv"),
    "vgg16_inference_timing.json": os.path.join(PHASE4_DIR, "vgg16_inference_timing.json"),
    "vgg16_evidence_notes.json": os.path.join(PHASE4_DIR, "vgg16_evidence_notes.json"),
    "vgg16_final_training_history.csv": os.path.join(PHASE4_DIR, "vgg16_final_training_history.csv"),
    "selected_vgg16_configuration.json": os.path.join(PHASE3_DIR, "selected_vgg16_configuration.json"),
    "vgg16_comparison_row.csv": comparison_csv_path,
    "vgg16_reproducibility_config.json": repro_config_path,
    "artifact_checksums.json": checksums_json_path,
    "MODEL_CARD.md": model_card_path,
}

missing_handoff = []

for filename, source_path in handoff_sources.items():
    if os.path.isfile(source_path):
        shutil.copy2(source_path, os.path.join(HANDOFF_DIR, filename))
    else:
        missing_handoff.append(source_path)

assert not missing_handoff, (
    "Cannot build final handoff because these files are missing:\n"
    + "\n".join(missing_handoff)
)

handoff_zip_base = os.path.join(RESULTS_DIR, "vgg16_handoff_lightweight")
handoff_zip_path = shutil.make_archive(
    handoff_zip_base,
    "zip",
    HANDOFF_DIR,
)

required_final_files = {
    "Final model": final_model_keras_path,
    "Final metrics": os.path.join(PHASE4_DIR, "vgg16_final_metrics.csv"),
    "Comparison row": comparison_csv_path,
    "Model card": model_card_path,
    "Reproducibility config": repro_config_path,
    "Lightweight handoff ZIP": handoff_zip_path,
}

print("\n" + "=" * 70)
print("MEMBER 2 — VGG16 — FINAL COMPLETION SUMMARY")
print("=" * 70)
print(f"Selected experiment : {m['selected_experiment']}")
print(f"Test accuracy       : {m['test_accuracy']:.6f}")
print(f"Macro F1            : {m['macro_f1']:.6f}")
print(f"Macro ROC-AUC       : {m['roc_auc_macro']:.6f}")
print(f"Total parameters    : {int(m['total_parameters']):,}")
print(f"Model size (MB)     : {m['model_size_mb']:.2f}")
print(f"Inference ms/image  : {m['inference_ms_per_image']:.4f}")
print()

for label, path in required_final_files.items():
    exists = os.path.isfile(path)
    print(f"{label:<24}: {exists}")
    assert exists, f"Missing required final artifact: {path}"

print(f"\nHandoff ZIP: {handoff_zip_path}")
print("\nMEMBER 2 STATUS: COMPLETE")


---
## Member 2 (VGG16) — Complete & Verified Handoff

All five phases for Member 2 are complete:
- **Phase 1:** Dataset audit verified, frozen split checksums matched, VGG16-specific preprocessing established.
- **Phase 2:** Pretrained VGG16 baseline constructed, classifier head justified, 1-epoch smoke test passed.
- **Phase 3:** Controlled experiments (VGG16-A, VGG16-B, VGG16-C) completed on validation data only; winning model selected and frozen.
- **Phase 4:** Held-out test set evaluated; all metrics, ROC-AUC, standardized latency, and error analysis exported without post-test tuning.
- **Phase 5:** Standardized comparison row, reproducibility configuration, artifact hashes, and model card packaged for group integration.

**Ready for handoff to Member 3 (ResNet50).**